In [11]:
# =========================
# Cell 1 — VALIDATION ONLY (no writes, no new columns)
# =========================
# - Verifies presence of required fields on nrel_db
# - Verifies types/values are suitable for modeling (numeric, 0/1, categorical)
# - Lists constraint fields and target values (subclass totals + appliance targets)
# - Shows filter ranges (energy min/max + optional filters) used for candidate selection
#
# NOTE: This cell DOES NOT edit any sheets or add columns. It only reads & reports.

import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

# -----------------------
# USER CONFIG
# -----------------------
excel_path = r"C:\Users\dmitr\Article_02\nrel_weights/weight_solver_05.xlsx"   # <-- change if needed

# hard expectations for schema
DB_REQUIRED = [
    "bldg_id",
    "out.electricity.total.energy_consumption",
    "in.geometry_building_type_recs",
    "in.bedrooms",
    "in.heating_fuel",
    "in.hvac_cooling_type",           # expected 0/1
    "in.clothes_washer_presence",     # expected 0/1
    "in.misc_freezer",                # expected 0/1
    "in.dishwasher",                  # expected 0/1
]

ARCH_REQUIRED = [
    "sub_class_id",
    "HousingStock",
    "annual_consumption_kwh",
    "out.electricity.total.energy_consumption_min",
    "out.electricity.total.energy_consumption_max",
    # optional filters for candidate selection
    "in.geometry_building_type_recs",
    "in.bedrooms",
    "in.heating_fuel",
]

APPS_REQUIRED = ["field", "op", "value", "target_pct"]
APPLIANCE_FIELDS_ALLOWED = [
    "in.hvac_cooling_type",
    "in.clothes_washer_presence",
    "in.misc_freezer",
    "in.dishwasher",
]
VALID_OPS = {"=", "==", "<>", "!="}

# -----------------------
# HELPERS
# -----------------------
def to_num(x):
    """Coerce Excel-ish numeric strings to float; return NaN if not parseable."""
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x)
    s = str(x).replace(",", "").strip()
    if s == "" or s.lower() in {"none", "nan"}:
        return np.nan
    try:
        return float(s)
    except Exception:
        return np.nan

def numeric_check(series, name, allow_na=False):
    """Return dict with numeric coercion stats."""
    vals = series.apply(to_num)
    n = len(vals)
    na = int(vals.isna().sum())
    bad = na if not allow_na else 0
    out = {
        "field": name,
        "numeric_like": int((vals.notna().sum()) == n) if not allow_na else int(True),
        "null_count": na,
        "null_pct": float(na / n * 100 if n else np.nan),
        "min": float(vals.min()) if vals.notna().any() else np.nan,
        "mean": float(vals.mean()) if vals.notna().any() else np.nan,
        "max": float(vals.max()) if vals.notna().any() else np.nan,
    }
    if (not allow_na) and na > 0:
        out["issue"] = "Non-numeric / null values present"
    return out

def zero_one_check(series, name):
    """Check values are in {0,1} (after numeric coercion)."""
    vals = series.apply(to_num)
    unique = sorted(set([v for v in vals.dropna().unique().tolist()]))
    ok = all(v in (0.0, 1.0) for v in unique)
    return {
        "field": name,
        "ok_zero_one": bool(ok),
        "unique_values": unique[:10],
        "null_count": int(vals.isna().sum()),
    }

def categorical_check(series, name, examples=8):
    s = series.astype(str).str.strip()
    n = len(s)
    nulls = int(series.isna().sum())
    ex = s.dropna().str.lower().drop_duplicates().head(examples).tolist()
    return {
        "field": name,
        "dtype": str(series.dtype),
        "null_count": nulls,
        "null_pct": float(nulls / n * 100 if n else np.nan),
        "examples": ex,
    }

# -----------------------
# LOAD SHEETS
# -----------------------
xls = pd.ExcelFile(excel_path, engine="openpyxl")
arch = pd.read_excel(xls, sheet_name="archetypes")
apps = pd.read_excel(xls, sheet_name="appliances")
db   = pd.read_excel(xls, sheet_name="nrel_db")

print("✓ Loaded sheets: archetypes, appliances, nrel_db")
print(f"nrel_db rows: {len(db)}\n")

# -----------------------
# 1) PRESENCE CHECK — nrel_db
# -----------------------
missing_db = [c for c in DB_REQUIRED if c not in db.columns]
if missing_db:
    print("✗ Missing on nrel_db:", missing_db)
else:
    print("✓ All required nrel_db columns are present.")

# -----------------------
# 2) TYPE / VALUE CHECKS — suitability for modeling
# -----------------------

# 2.1 nrel_db: ID & uniqueness
if "bldg_id" in db.columns:
    nunq = db["bldg_id"].nunique(dropna=True)
    print(f"\n[bldg_id] unique={nunq} / rows={len(db)} ; nulls={db['bldg_id'].isna().sum()}")

# 2.2 nrel_db: numeric checks
num_reports = []
for col in [
    "out.electricity.total.energy_consumption",
    "in.bedrooms",
]:
    if col in db.columns:
        num_reports.append(numeric_check(db[col], col, allow_na=False))
if num_reports:
    print("\nNumeric checks (nrel_db):")
    display(pd.DataFrame(num_reports))

# 2.3 nrel_db: 0/1 checks for appliance/cooling presence
zero_one_reports = []
for col in [
    "in.hvac_cooling_type",
    "in.clothes_washer_presence",
    "in.misc_freezer",
    "in.dishwasher",
]:
    if col in db.columns:
        zero_one_reports.append(zero_one_check(db[col], col))
if zero_one_reports:
    print("Zero/One (0/1) checks (nrel_db):")
    display(pd.DataFrame(zero_one_reports))

# 2.4 nrel_db: categorical checks for filters
cat_reports = []
for col in ["in.geometry_building_type_recs", "in.heating_fuel"]:
    if col in db.columns:
        cat_reports.append(categorical_check(db[col], col))
if cat_reports:
    print("Categorical checks (nrel_db):")
    display(pd.DataFrame(cat_reports))

# 2.5 archetypes: numeric constraints & ranges
arch_num_reports = []
for col in [
    "HousingStock",
    "annual_consumption_kwh",
    "out.electricity.total.energy_consumption_min",
    "out.electricity.total.energy_consumption_max",
]:
    if col in arch.columns:
        arch_num_reports.append(numeric_check(arch[col], col, allow_na=False))
if arch_num_reports:
    print("\nNumeric checks (archetypes):")
    display(pd.DataFrame(arch_num_reports))

# 2.6 archetypes: optional filters (strings/numbers as-is)
arch_cat_reports = []
for col in ["in.geometry_building_type_recs", "in.heating_fuel"]:
    if col in arch.columns:
        arch_cat_reports.append(categorical_check(arch[col], col))
if "in.bedrooms" in arch.columns:
    # bedrooms can be tokens like '4 or 5' or integers; we just show examples
    arch_cat_reports.append(categorical_check(arch["in.bedrooms"], "in.bedrooms"))
if arch_cat_reports:
    print("Filter columns (archetypes) — examples:")
    display(pd.DataFrame(arch_cat_reports))

# 2.7 appliances: ops/fields/values/targets
apps_issues = []
if set(APPS_REQUIRED) - set(apps.columns):
    apps_issues.append("Missing required columns on 'appliances'.")
invalid_ops = sorted(set(apps["op"].astype(str).str.strip()) - VALID_OPS)
if invalid_ops:
    apps_issues.append(f"Invalid operators found: {invalid_ops} (allowed: {sorted(VALID_OPS)})")
invalid_fields = sorted(set(apps["field"].dropna().unique()) - set(APPLIANCE_FIELDS_ALLOWED))
if invalid_fields:
    apps_issues.append(f"'appliances.field' includes unsupported columns: {invalid_fields}")

# Targets: accept both [0..1] fractions or [0..100] percentages; normalize for reporting
apps_target_raw = apps["target_pct"].apply(to_num)
if apps_target_raw.max() is not np.nan and apps_target_raw.max() is not None:
    if float(apps_target_raw.max()) <= 1.0:
        apps_target_norm = apps_target_raw.clip(lower=0, upper=1)
    else:
        apps_target_norm = (apps_target_raw / 100.0).clip(lower=0, upper=1)
else:
    apps_target_norm = apps_target_raw

apps_report = pd.DataFrame({
    "field": apps["field"],
    "op": apps["op"].astype(str).str.strip(),
    "value": apps["value"],
    "target_raw": apps["target_pct"],
    "target_norm_0to1": apps_target_norm,
})

print("\nAppliances sheet checks:")
if apps_issues:
    for msg in apps_issues:
        print("✗", msg)
else:
    print("✓ Operators & field names look OK.")
display(apps_report)

# -----------------------
# 3) LIST CONSTRAINTS & TARGETS
# -----------------------

# 3.1 Subclass totals (constraints): H and Total Energy, plus implied E*
arch_constraints = arch.copy()
arch_constraints["HousingStock_num"] = arch_constraints["HousingStock"].apply(to_num)
arch_constraints["annual_consumption_kwh_num"] = arch_constraints["annual_consumption_kwh"].apply(to_num)
arch_constraints["E_star_kwh_per_HH"] = (
    arch_constraints["annual_consumption_kwh_num"] / arch_constraints["HousingStock_num"]
)

constraint_cols = [
    "sub_class_id",
    "HousingStock_num",
    "annual_consumption_kwh_num",
    "E_star_kwh_per_HH",
]
print("\nConstraint targets — subclass totals (H, TotalEnergy, E*):")
display(arch_constraints[constraint_cols].head(20))

# 3.2 Appliance penetration targets (normalized to 0..1)
print("Constraint targets — appliance penetrations (normalized to 0..1):")
display(apps_report[["field", "op", "value", "target_norm_0to1"]])

# 3.3 Candidate-selection filters (not constraints, but important to sanity-check)
filters_preview = arch[[
    "sub_class_id",
    "out.electricity.total.energy_consumption_min",
    "out.electricity.total.energy_consumption_max",
    "in.geometry_building_type_recs",
    "in.bedrooms",
    "in.heating_fuel",
]].copy()
print("\nCandidate-selection filters (energy range + optional filters):")
display(filters_preview.head(20))

# -----------------------
# SUMMARY FLAGS
# -----------------------
print("\n=== SUMMARY FLAGS ===")
if missing_db:
    print("✗ Fix: add missing nrel_db fields:", missing_db)

# Numeric viability
nrel_num_ok = all(r.get("numeric_like", True) for r in num_reports)
arch_num_ok = all(r.get("numeric_like", True) for r in arch_num_reports)
if not nrel_num_ok:
    print("✗ Fix: some nrel_db numeric fields failed coercion.")
if not arch_num_ok:
    print("✗ Fix: some archetypes numeric fields failed coercion.")

# Zero/one checks
z01_df = pd.DataFrame(zero_one_reports) if zero_one_reports else pd.DataFrame()
if not z01_df.empty and not z01_df["ok_zero_one"].all():
    offenders = z01_df.loc[~z01_df["ok_zero_one"], "field"].tolist()
    print("✗ Fix: 0/1 expectation violated in:", offenders)
else:
    print("✓ 0/1 expectation satisfied for appliance/cooling presence fields.")

# Appliances sheet OK?
if apps_issues:
    print("✗ Fix issues in 'appliances' listed above.")
else:
    print("✓ Appliances table looks consistent (fields/ops/targets).")

print("✓ Validation complete (no data was modified).")


✓ Loaded sheets: archetypes, appliances, nrel_db
nrel_db rows: 14016

✓ All required nrel_db columns are present.

[bldg_id] unique=14016 / rows=14016 ; nulls=0

Numeric checks (nrel_db):


,field,numeric_like,null_count,null_pct,min,mean,max
0,out.electricity.total.energy_consumption,1,0,0.0,1267.570188,9874.634980,118828.9084
1,in.bedrooms,1,0,0.0,1.000000,2.680151,5.0000


Zero/One (0/1) checks (nrel_db):


,field,ok_zero_one,unique_values,null_count
0,in.hvac_cooling_type,True,"[0.0, 1.0]",0
1,in.clothes_washer_presence,True,"[0.0, 1.0]",0
2,in.misc_freezer,True,"[0.0, 1.0]",0
3,in.dishwasher,True,"[0.0, 1.0]",0


Categorical checks (nrel_db):


...

Constraint targets — appliance penetrations (normalized to 0..1):


,field,op,value,target_norm_0to1
0,in.hvac_cooling_type,=,1,0.20
1,in.clothes_washer_presence,=,1,0.92
2,in.misc_freezer,=,1,0.12
3,in.dishwasher,=,1,0.10



Candidate-selection filters (energy range + optional filters):


,sub_class_id,out.electricity.total.energy_consumption_min,out.electricity.total.energy_consumption_max,in.geometry_building_type_recs,in.bedrooms,in.heating_fuel
0,Subclass_01,0,2400,any,any,Electricity
1,Subclass_02,2400,6000,any,any,Electricity
2,Subclass_03,6000,9600,any,any,Electricity
3,Subclass_04,9600,14400,any,any,Electricity
4,Subclass_05,144000,144000,any,any,Electricity
5,Subclass_06,28800,999999,any,any,Electricity
6,Subclass_07,0,1200,"""Multi-Family with 2 - 4 Units"" or ""Multi-Fami...",1,any
7,Subclass_08,0,1200,"""Multi-Family with 2 - 4 Units"" or ""Multi-Fami...",2,any
8,Subclass_09,0,1200,"""Multi-Family with 2 - 4 Units"" or ""Multi-Fami...",3,any
9,Subclass_10,0,1200,"""Multi-Family with 2 - 4 Units"" or ""Multi-Fami...",4 or 5,any



=== SUMMARY FLAGS ===
✓ 0/1 expectation satisfied for appliance/cooling presence fields.
✓ Appliances table looks consistent (fields/ops/targets).
✓ Validation complete (no data was modified).


In [2]:
# =========================
# Cell 2 — OPTIMIZE A RANGE OF SUBCLASSES AND SAVE TO NEW FILE
# =========================
# - Selects subclasses by numeric range, e.g., Subclass_01 .. Subclass_10
# - Runs the QP per selected subclass only
# - Writes 'weights' and 'weights_summary' to a NEW Excel file:
#   e.g., "<originalname>__weights_Subclass_01_to_Subclass_10.xlsx"
#
# Assumes: archetypes/appliances/nrel_db tables follow the schema validated earlier.

import re
import time
import numpy as np
import pandas as pd
from pathlib import Path
from pyomo.environ import (
    ConcreteModel, Var, Objective, Constraint, NonNegativeReals,
    SolverFactory, value, Set as PyomoSet
)

# -----------------------
# USER CONFIG
# -----------------------
excel_path = r"C:\Users\dmitr\Article_02\nrel_weights/weight_solver_11.xlsx"  # source workbook
# Choose range (inclusive) by subclass numeric code in the tail of 'sub_class_id'
FROM_SUBCLASS = "Subclass_13"
TO_SUBCLASS   = "Subclass_14"

# If None, we auto-name: "<stem>__weights_<FROM>_to_<TO>.xlsx"
save_as = None

# Solver config
SOLVER        = "ipopt"
IPOPT_EXE     = ""                # e.g., r"C:/.../ipopt.exe" or leave "" to rely on PATH
SOLVER_TMAX_S = 60

# Objective weights
alpha = 1.0   # energy squared deviation
beta  = 0.5   # appliance penetrations squared deviation
gamma = 1e-3  # L2 regularization on w

# Sum band
sum_band_eps = 0.01   # ±1%

# Weight bounds
rho_min_weight   = 0.10        # floor: L_k = rho * H/N
wmax_fraction_of_H = None      # e.g., 0.15 to cap; None for no cap

# -----------------------
# HELPERS
# -----------------------
def to_num(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x)
    s = str(x).replace(",", "").strip()
    if s == "" or s.lower() in {"none", "nan"}:
        return np.nan
    try:
        return float(s)
    except Exception:
        return np.nan

def parse_building_type_filter(txt):
    if pd.isna(txt):
        return None
    s = str(txt).strip()
    if s == "" or s.lower() == "any":
        return None
    if '"' in s:
        parts = [p.strip().strip('"') for p in s.split(" or ")]
        parts = [p for p in parts if p]
        return parts or None
    return [s]

def bedrooms_mask(series, token):
    if pd.isna(token):
        return pd.Series(True, index=series.index)
    t = str(token).strip().lower()
    if t == "" or t == "any":
        return pd.Series(True, index=series.index)
    if "or" in t:
        vals = [v.strip() for v in t.split("or")]
        want = set()
        for v in vals:
            vv = to_num(v)
            if np.isfinite(vv):
                want.add(int(round(vv)))
        return series.round().astype("Int64").isin(list(want)).fillna(False)
    v = to_num(t)
    if np.isfinite(v):
        return series.round().astype("Int64").eq(int(round(v))).fillna(False)
    return pd.Series(True, index=series.index)

def auto_shrink_rho_for_feasibility(rho, H, N):
    sumL = rho * H / max(N, 1) * N
    if sumL <= H:
        return rho
    return (0.99 * H) / (H + 1e-12)

def effective_sample_size(w):
    w = np.asarray(w, dtype=float)
    s = w.sum()
    if s <= 0:
        return 0.0
    return float((s * s) / (np.sum(w * w) + 1e-12))

def gini_coef(w):
    w = np.asarray(w, dtype=float)
    if w.size == 0 or np.allclose(w, 0):
        return 0.0
    sw = np.sort(w)
    n  = w.size
    cum = np.cumsum(sw)
    return float((n + 1 - 2 * np.sum(cum) / cum[-1]) / n)

def get_solver():
    s = SolverFactory(SOLVER, executable=IPOPT_EXE) if (IPOPT_EXE and Path(IPOPT_EXE).exists()) else SolverFactory(SOLVER)
    try:
        s.options["max_cpu_time"] = float(SOLVER_TMAX_S)
        s.options["tol"] = 1e-6
        s.options["constr_viol_tol"] = 1e-6
    except Exception:
        pass
    return s

def subclass_num(sid: str):
    """Extract trailing integer from 'Subclass_xx'. Returns int or None."""
    m = re.search(r'(\d+)$', str(sid))
    return int(m.group(1)) if m else None

# -----------------------
# LOAD INPUTS
# -----------------------
xls = pd.ExcelFile(excel_path, engine="openpyxl")
arch = pd.read_excel(xls, sheet_name="archetypes")
apps = pd.read_excel(xls, sheet_name="appliances")
db   = pd.read_excel(xls, sheet_name="nrel_db")

# Normalize numeric for computation
arch["_Emin"] = arch["out.electricity.total.energy_consumption_min"].apply(to_num)
arch["_Emax"] = arch["out.electricity.total.energy_consumption_max"].apply(to_num)
arch["_H"]    = arch["HousingStock"].apply(to_num)
arch["_TotE"] = arch["annual_consumption_kwh"].apply(to_num)

db["_E"]   = pd.to_numeric(db["out.electricity.total.energy_consumption"].apply(to_num), errors="coerce")
db["_bed"] = pd.to_numeric(db["in.bedrooms"].apply(to_num), errors="coerce")

# Appliances target normalization (accept 0..1 or 0..100)
apps["_target_raw"] = apps["target_pct"].apply(to_num)
apps["_target"] = (
    apps["_target_raw"].where(apps["_target_raw"] <= 1.0, apps["_target_raw"] / 100.0)
).clip(lower=0, upper=1)

# -----------------------
# SELECT SUBCLASS RANGE
# -----------------------
from_n = subclass_num(FROM_SUBCLASS)
to_n   = subclass_num(TO_SUBCLASS)
if from_n is None or to_n is None:
    raise ValueError(f"Could not parse subclass numbers from: {FROM_SUBCLASS}, {TO_SUBCLASS}")
if from_n > to_n:
    from_n, to_n = to_n, from_n

arch["_sid_num"] = arch["sub_class_id"].astype(str).map(subclass_num)
sel_mask = arch["_sid_num"].between(from_n, to_n, inclusive="both")
arch_sel = arch[sel_mask].copy().reset_index(drop=True)

if arch_sel.empty:
    raise RuntimeError(f"No archetypes found in range Subclass_{from_n:02d}..Subclass_{to_n:02d}")

print(f"Running subclasses: {arch_sel['sub_class_id'].tolist()}")

# -----------------------
# PREP OUTPUT FRAMES
# -----------------------
all_bldg_ids = db["bldg_id"].tolist()
weights_out  = pd.DataFrame({"bldg_id": all_bldg_ids})
summary_rows = []

solver = get_solver()

# -----------------------
# MAIN LOOP (RANGE ONLY)
# -----------------------
for _, arow in arch_sel.iterrows():
    sid = str(arow["sub_class_id"])
    t0 = time.perf_counter()

    # Targets
    H = int(round(arow["_H"])) if np.isfinite(arow["_H"]) else 0
    TotE = float(arow["_TotE"]) if np.isfinite(arow["_TotE"]) else np.nan

    if H <= 0:
        weights_out[sid] = 0.0
        summary_rows.append({
            "sub_class_id": sid, "status": "skipped_H<=0",
            "H_target": H, "H_actual": 0.0,
            "TotalEnergy_target_kwh": TotE,
            "TotalEnergy_actual_kwh": 0.0,
            "energy_rel_error_pct": np.nan,
            "candidates_N": 0
        })
        continue

    # Energy band
    E_min = float(arow["_Emin"]) if np.isfinite(arow["_Emin"]) else -np.inf
    E_max = float(arow["_Emax"]) if np.isfinite(arow["_Emax"]) else  np.inf
    if E_min >= E_max:
        E_min, E_max = -np.inf, np.inf

    # ---- Candidate filters ----
    cand = db[(db["_E"] >= E_min) & (db["_E"] < E_max)].copy()

    # building type
    bt_list = parse_building_type_filter(arow.get("in.geometry_building_type_recs"))
    if bt_list:
        bt_lower = {b.lower() for b in bt_list}
        cand = cand[cand["in.geometry_building_type_recs"].astype(str).str.strip().str.lower().isin(bt_lower)]

    # bedrooms
    mask_bed = bedrooms_mask(cand["_bed"], arow.get("in.bedrooms"))
    cand = cand[mask_bed]

    # heating fuel
    hf = arow.get("in.heating_fuel")
    if isinstance(hf, str) and hf.strip().lower() != "any" and hf.strip() != "":
        cand = cand[cand["in.heating_fuel"].astype(str).str.strip().str.lower() == hf.strip().lower()]

    N = len(cand)
    if N == 0:
        weights_out[sid] = 0.0
        summary_rows.append({
            "sub_class_id": sid, "status": "no_candidates_after_filter",
            "H_target": H, "H_actual": 0.0,
            "TotalEnergy_target_kwh": TotE,
            "TotalEnergy_actual_kwh": 0.0,
            "energy_rel_error_pct": np.nan,
            "candidates_N": 0,
            "E_min": E_min, "E_max": E_max,
            "bt_filter": bt_list, "bedrooms_filter": arow.get("in.bedrooms"),
            "heating_fuel_filter": arow.get("in.heating_fuel")
        })
        continue

    # Data vectors
    E = cand["_E"].to_numpy(dtype=float)

    # Appliance presence matrix (0/1 already in DB)
    I_cols = []
    for _, r in apps.iterrows():
        fld  = str(r["field"]).strip()
        op   = str(r["op"]).strip()
        val  = to_num(r["value"])
        if fld not in cand.columns:
            raise RuntimeError(f"Appliance field '{fld}' not in nrel_db.")
        x = pd.to_numeric(cand[fld].apply(to_num), errors="coerce").fillna(0.0).to_numpy(dtype=float)
        if op in ("=", "=="):
            if val == 1.0:
                col = x
            elif val == 0.0:
                col = 1.0 - x
            else:
                col = (np.isclose(x, float(val))).astype(float)
        elif op in ("<>", "!="):
            if val == 1.0:
                col = 1.0 - x
            elif val == 0.0:
                col = x
            else:
                col = (~np.isclose(x, float(val))).astype(float)
        else:
            raise RuntimeError(f"Unsupported operator '{op}' in appliances.")
        I_cols.append(col)
    I = np.vstack(I_cols).T
    Pen = apps["_target"].to_numpy(dtype=float)
    M = I.shape[1]

    # Priorities & bounds
    p = np.ones(N, dtype=float)
    rho = auto_shrink_rho_for_feasibility(float(rho_min_weight), H, N)
    L = rho * H / max(N, 1) * p
    U = np.full(N, wmax_fraction_of_H * H, dtype=float) if (wmax_fraction_of_H is not None) else np.full(N, float('inf'))

    # Sum band
    H_lo = (1.0 - float(sum_band_eps)) * float(H)
    H_hi = (1.0 + float(sum_band_eps)) * float(H)

    # Energy per HH target
    E_star = (TotE / H) if (np.isfinite(TotE) and H > 0) else np.nan

    # ---- Build & solve model ----
    model = ConcreteModel()
    model.K = PyomoSet(initialize=list(range(N)))

    def w_bounds(m, k):
        return (float(L[k]), float(U[k]))
    model.w = Var(model.K, domain=NonNegativeReals, bounds=w_bounds)

    if sum_band_eps <= 0.0:
        def sum_w_eq(m): return sum(m.w[k] for k in m.K) == float(H)
        model.sum_w_eq = Constraint(rule=sum_w_eq)
    else:
        model.sum_w_lo = Constraint(expr=sum(model.w[k] for k in model.K) >= H_lo)
        model.sum_w_hi = Constraint(expr=sum(model.w[k] for k in model.K) <= H_hi)

    def objective_rule(m):
        energy_target = float(H) * (0.0 if not np.isfinite(E_star) else float(E_star))
        energy_dev = (sum(m.w[k] * E[k] for k in m.K) - energy_target)**2
        app_part = 0.0
        for j in range(M):
            frac_j = sum(m.w[k] * I[k, j] for k in m.K) / float(H)
            app_part += (frac_j - float(Pen[j]))**2
        reg = sum(m.w[k]**2 for k in m.K)
        return float(alpha)*energy_dev + float(beta)*app_part + float(gamma)*reg
    model.obj = Objective(rule=objective_rule)

    try:
        res = get_solver().solve(model, tee=False)
        solver_status = str(res.solver.status)
        solver_term   = str(res.solver.termination_condition)
    except Exception as e:
        solver_status = "exception"
        solver_term   = f"{e}"

    # Extract & map
    w_opt = np.array([value(model.w[k]) if model.w[k].value is not None else 0.0 for k in model.K])
    col = np.zeros(len(db), dtype=float)
    col[cand.index.to_numpy()] = w_opt
    weights_out[sid] = col

    # Diagnostics
    sum_w = float(np.nansum(w_opt))
    active = int(np.sum(w_opt > 0))
    TotalEnergy_actual = float(np.dot(w_opt, E))
    energy_rel_error = ((TotalEnergy_actual - TotE) / TotE * 100.0) if (np.isfinite(TotE) and TotE != 0) else np.nan

    if M > 0 and H > 0:
        actual_pens = (w_opt[:, None] * I).sum(axis=0) / float(H)
        app_diffs = actual_pens - Pen
        app_mae = float(np.mean(np.abs(app_diffs)))
        app_mse = float(np.mean(app_diffs**2))
        app_max_abs = float(np.max(np.abs(app_diffs)))
    else:
        app_mae = app_mse = app_max_abs = np.nan

    _E_star_val = 0.0 if not np.isfinite(E_star) else float(E_star)
    obj_energy = float((np.dot(w_opt, E) - float(H)*_E_star_val)**2)
    obj_apps   = float(np.sum(((actual_pens - Pen) if (M>0 and H>0) else np.zeros(0))**2))
    obj_reg    = float(np.sum(w_opt**2))
    obj_total  = float(alpha*obj_energy + beta*obj_apps + gamma*obj_reg)

    if N > 0 and sum_w > 0:
        ws = np.sort(w_opt)[::-1]
        top1_share = float(ws[0] / sum_w)
        top5_share = float(ws[:min(5, N)].sum() / sum_w)
    else:
        top1_share = top5_share = np.nan

    t1 = time.perf_counter()

    summary_rows.append({
        "sub_class_id": sid,
        "status": f"{solver_status}/{solver_term}",
        "solve_sec": t1 - t0,
        "candidates_N": int(N),
        "H_target": int(H),
        "H_lo": H_lo, "H_hi": H_hi,
        "sum_w": sum_w,
        "E_min": E_min, "E_max": E_max,
        "TotalEnergy_target_kwh": float(TotE) if np.isfinite(TotE) else np.nan,
        "TotalEnergy_actual_kwh": float(TotalEnergy_actual),
        "energy_rel_error_pct": float(energy_rel_error) if np.isfinite(energy_rel_error) else np.nan,
        "rho_used": float(rho),
        "L_min": float(np.min(L)), "L_max": float(np.max(L)),
        "U_min": float(np.min(U)), "U_max": float(np.max(U)),
        "objective_value": obj_total,
        "obj_energy": obj_energy,
        "obj_apps": obj_apps,
        "obj_reg": obj_reg,
        "appliance_mae_abs": app_mae,
        "appliance_mse": app_mse,
        "appliance_max_abs_diff": app_max_abs,
        "active_archetypes_w>0": active,
        "top1_share": top1_share,
        "top5_share": top5_share,
        "N_eff": effective_sample_size(w_opt),
        "gini": gini_coef(w_opt),
        "bt_filter": "; ".join(bt_list) if bt_list else "any",
        "bedrooms_filter": str(arow.get("in.bedrooms")),
        "heating_fuel_filter": str(arow.get("in.heating_fuel")),
    })

# -----------------------
# SAVE TO NEW FILE
# -----------------------
stem = Path(excel_path).with_suffix("").name
default_name = f"{stem}_weights_{FROM_SUBCLASS}_to_{TO_SUBCLASS}.xlsx"
out_path = Path(excel_path).with_name(save_as or default_name)

with pd.ExcelWriter(out_path, engine="openpyxl", mode="w") as writer:
    weights_out.to_excel(writer, sheet_name="weights", index=False)
    pd.DataFrame(summary_rows).to_excel(writer, sheet_name="weights_summary", index=False)

print(f"Done. Wrote range results to: {out_path}")


Running subclasses: ['Subclass_13', 'Subclass_14']
Done. Wrote range results to: C:\Users\dmitr\Article_02\nrel_weights\weight_solver_11_weights_Subclass_13_to_Subclass_14.xlsx


In [1]:
# =========================
# Cell 3 — CANDIDATE COUNTS + PENETRATIONS PER SUBCLASS
#           + OVERALL APPLIANCE PENETRATIONS VS TARGETS
# =========================
# Output: NEW Excel with two sheets:
#   1) 'candidates_count' — per subclass: filters, candidate count, 4 appliance penetrations (0..1)
#   2) 'appliance_totals_vs_targets' — overall sample penetrations (0..1) vs targets
#
# Read-only w.r.t. the source workbook.

import numpy as np
import pandas as pd
from pathlib import Path

# -----------------------
# USER CONFIG
# -----------------------
excel_path = r"C:\Users\dmitr\Article_02\nrel_weights/weight_solver_15i.xlsx"   # source workbook
# If None, auto-name: "<stem>__candidates_count.xlsx"
save_as = None

# -----------------------
# HELPERS (match optimization logic)
# -----------------------
def to_num(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x)
    s = str(x).replace(",", "").strip()
    if s == "" or s.lower() in {"none", "nan"}:
        return np.nan
    try:
        return float(s)
    except Exception:
        return np.nan

def parse_building_type_filter(txt):
    """
    Accepts:
      - any
      - "A" or "B"
      - Single-Family Detached
    Returns list of allowed labels (case-insensitive) or None for 'any'/blank.
    """
    if pd.isna(txt):
        return None
    s = str(txt).strip()
    if s == "" or s.lower() == "any":
        return None
    if '"' in s:
        parts = [p.strip().strip('"') for p in s.split(" or ")]
        parts = [p for p in parts if p]
        return parts or None
    return [s]

def bedrooms_mask(series_numeric, token):
    """
    series_numeric: numeric bedrooms series (float/int)
    token in archetypes: int like 1/2/3, string '4 or 5', 'any', or blank.
    """
    if pd.isna(token):
        return pd.Series(True, index=series_numeric.index)
    t = str(token).strip().lower()
    if t == "" or t == "any":
        return pd.Series(True, index=series_numeric.index)
    if "or" in t:
        vals = [v.strip() for v in t.split("or")]
        want = set()
        for v in vals:
            vv = to_num(v)
            if np.isfinite(vv):
                want.add(int(round(vv)))
        return series_numeric.round().astype("Int64").isin(list(want)).fillna(False)
    v = to_num(t)
    if np.isfinite(v):
        return series_numeric.round().astype("Int64").eq(int(round(v))).fillna(False)
    return pd.Series(True, index=series_numeric.index)

def penetration_0to1(df, col):
    """
    Safe penetration (mean) for a 0/1 column; returns NaN if df is empty.
    Coerces to numeric and clips to [0,1] before taking the mean.
    """
    if df.empty or col not in df.columns:
        return np.nan
    s = pd.to_numeric(df[col].apply(to_num), errors="coerce").clip(lower=0, upper=1)
    return float(s.mean()) if s.notna().any() else np.nan

# -----------------------
# LOAD INPUTS
# -----------------------
xls = pd.ExcelFile(excel_path, engine="openpyxl")
arch = pd.read_excel(xls, sheet_name="archetypes")
apps = pd.read_excel(xls, sheet_name="appliances")
db   = pd.read_excel(xls, sheet_name="nrel_db")

# Precompute numeric columns in memory (no writes to file)
db["_E"]   = pd.to_numeric(db["out.electricity.total.energy_consumption"].apply(to_num), errors="coerce")
db["_bed"] = pd.to_numeric(db["in.bedrooms"].apply(to_num), errors="coerce")

arch["_Emin"] = arch["out.electricity.total.energy_consumption_min"].apply(to_num)
arch["_Emax"] = arch["out.electricity.total.energy_consumption_max"].apply(to_num)

APPLIANCE_FIELDS = [
    "in.hvac_cooling_type",
    "in.clothes_washer_presence",
    "in.misc_freezer",
    "in.dishwasher",
]

# -----------------------
# 1) PER-SUBCLASS: counts + penetrations
# -----------------------
rows = []
for _, a in arch.iterrows():
    sid   = str(a["sub_class_id"])
    sname = str(a.get("sub_class", ""))

    # energy band (fail-open if misconfigured)
    E_min = float(a["_Emin"]) if np.isfinite(a["_Emin"]) else -np.inf
    E_max = float(a["_Emax"]) if np.isfinite(a["_Emax"]) else  np.inf
    if E_min >= E_max:
        E_min, E_max = -np.inf, np.inf

    # start with energy filter
    cand = db[(db["_E"] >= E_min) & (db["_E"] < E_max)]

    # building type
    bt_list = parse_building_type_filter(a.get("in.geometry_building_type_recs"))
    if bt_list:
        bt_lower = {b.lower() for b in bt_list}
        cand = cand[cand["in.geometry_building_type_recs"].astype(str).str.strip().str.lower().isin(bt_lower)]

    # bedrooms
    mask_bed = bedrooms_mask(cand["_bed"], a.get("in.bedrooms"))
    cand = cand[mask_bed]

    # heating fuel
    hf = a.get("in.heating_fuel")
    if isinstance(hf, str) and hf.strip().lower() != "any" and hf.strip() != "":
        cand = cand[cand["in.heating_fuel"].astype(str).str.strip().str.lower() == hf.strip().lower()]

    row = {
        "sub_class_id": sid,
        "sub_class": sname,
        "E_min": None if not np.isfinite(E_min) else E_min,
        "E_max": None if not np.isfinite(E_max) else E_max,
        "filter_building_type": "any" if not bt_list else " or ".join(bt_list),
        "filter_bedrooms": str(a.get("in.bedrooms")),
        "filter_heating_fuel": str(a.get("in.heating_fuel")),
        "candidates_N": int(len(cand)),
    }
    # appliance penetrations (0..1)
    for fld in APPLIANCE_FIELDS:
        row[f"pen.{fld}"] = penetration_0to1(cand, fld)

    rows.append(row)

candidates_df = pd.DataFrame(rows, columns=[
    "sub_class_id","sub_class","E_min","E_max",
    "filter_building_type","filter_bedrooms","filter_heating_fuel",
    "candidates_N",
    "pen.in.hvac_cooling_type",
    "pen.in.clothes_washer_presence",
    "pen.in.misc_freezer",
    "pen.in.dishwasher",
])

# -----------------------
# 2) OVERALL: appliance penetrations vs targets
# -----------------------
# Compute overall penetrations on full nrel_db (0..1)
overall = []
for fld in APPLIANCE_FIELDS:
    overall.append({
        "field": fld,
        "overall_penetration_0to1": penetration_0to1(db, fld),
    })
overall_df = pd.DataFrame(overall)

# Normalize target_pct: accept 0..1 or 0..100
apps["_target_raw"] = apps["target_pct"].apply(to_num)
apps["_target_norm_0to1"] = (
    apps["_target_raw"].where(apps["_target_raw"] <= 1.0, apps["_target_raw"] / 100.0)
).clip(lower=0, upper=1)

targets_df = apps[["field", "op", "value", "target_pct"]].copy()
targets_df = targets_df.merge(
    apps[["field", "_target_norm_0to1"]],
    on="field", how="left"
).merge(
    overall_df, on="field", how="left"
)

targets_df["abs_diff"] = (targets_df["overall_penetration_0to1"] - targets_df["_target_norm_0to1"]).abs()

# Arrange columns nicely
targets_df = targets_df[[
    "field", "op", "value", "target_pct",
    "_target_norm_0to1", "overall_penetration_0to1", "abs_diff"
]].rename(columns={
    "_target_norm_0to1": "target_norm_0to1"
})

# -----------------------
# SAVE NEW EXCEL (two sheets)
# -----------------------
stem = Path(excel_path).with_suffix("").name
out_name = save_as or f"{stem}__candidates_count.xlsx"
out_path = Path(excel_path).with_name(out_name)

with pd.ExcelWriter(out_path, engine="openpyxl", mode="w") as writer:
    candidates_df.to_excel(writer, sheet_name="candidates_count", index=False)
    targets_df.to_excel(writer, sheet_name="appliance_totals_vs_targets", index=False)

print(f"Done. Wrote:\n - {out_path.name} [sheets: candidates_count, appliance_totals_vs_targets]")


Done. Wrote:
 - weight_solver_15i__candidates_count.xlsx [sheets: candidates_count, appliance_totals_vs_targets]


In [7]:
# =========================
# Cell 2 — OPTIMIZE A RANGE OF SUBCLASSES (ENERGY-ONLY) AND SAVE TO NEW FILE
# =========================
# - Runs ONLY energy matching per subclass (no appliance penetration terms)
# - Uses softened sum constraint: (1-ε)H <= Σw <= (1+ε)H
# - Writes 'weights' and 'weights_summary' to a NEW Excel file
#
# Assumes validation cell passed. This cell does not modify inputs.

import re
import time
import numpy as np
import pandas as pd
from pathlib import Path
from pyomo.environ import (
    ConcreteModel, Var, Objective, Constraint, NonNegativeReals,
    SolverFactory, value, Set as PyomoSet
)

# -----------------------
# USER CONFIG
# -----------------------
excel_path =  r"C:\Users\dmitr\Article_02\nrel_weights/weight_solver_13.xlsx"  # source workbook
FROM_SUBCLASS = "Subclass_01"   # inclusive
TO_SUBCLASS   = "Subclass_19"   # inclusive

# If None, auto-names "<stem>__weights_<FROM>_to_<TO>__energy_only.xlsx"
save_as = None

# Solver config
SOLVER        = "ipopt"
IPOPT_EXE     = ""               # set a path if ipopt.exe is not on PATH
SOLVER_TMAX_S = 60

# Objective weights (ENERGY-ONLY NOW)
alpha = 1.0    # energy squared deviation
gamma = 1e-5   # L2 regularization on w

# Sum band (soften Σw=H)
sum_band_eps = 0.03  # ±1%

# Weight bounds
rho_min_weight    = 0.0010         # floor: L_k = rho * H/N (auto-shrunk if needed)
wmax_fraction_of_H = None        # e.g., 0.15 to cap per-candidate; None to disable

# -----------------------
# HELPERS
# -----------------------
def to_num(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x)
    s = str(x).replace(",", "").strip()
    if s == "" or s.lower() in {"none", "nan"}:
        return np.nan
    try:
        return float(s)
    except Exception:
        return np.nan

def parse_building_type_filter(txt):
    """Allow 'any' or multiple options separated by 'or' (quotes optional)."""
    if txt is None:
        return None
    s = str(txt).strip()
    if s == "" or s.lower() == "any":
        return None
    parts = [p.strip().strip('"').strip("'") for p in re.split(r'\s+or\s+', s, flags=re.I)]
    parts = [p for p in parts if p]
    return parts or None

def bedrooms_mask(series, token):
    """Accepts: any / blank / '3' / '3 or 4 or 5' / '3 or 4 bedrooms'."""
    if token is None:
        return pd.Series(True, index=series.index)
    t = str(token).strip().lower()
    if t == "" or t == "any":
        return pd.Series(True, index=series.index)
    nums = [int(x) for x in re.findall(r'\d+', t)]
    if nums:
        want = set(nums)
        return series.round().astype("Int64").isin(list(want)).fillna(False)
    return pd.Series(True, index=series.index)

def auto_shrink_rho_for_feasibility(rho, H, N):
    sumL = rho * H / max(N, 1) * N
    if sumL <= H:
        return rho
    # shrink just enough to be feasible
    return (0.99 * H) / max(H, 1e-12)

def effective_sample_size(w):
    w = np.asarray(w, dtype=float)
    s = w.sum()
    if s <= 0:
        return 0.0
    return float((s * s) / (np.sum(w * w) + 1e-12))

def gini_coef(w):
    w = np.asarray(w, dtype=float)
    if w.size == 0 or np.allclose(w, 0):
        return 0.0
    sw = np.sort(w)
    n  = w.size
    cum = np.cumsum(sw)
    return float((n + 1 - 2 * np.sum(cum) / cum[-1]) / n)

def get_solver():
    s = SolverFactory(SOLVER, executable=IPOPT_EXE) if (IPOPT_EXE and Path(IPOPT_EXE).exists()) else SolverFactory(SOLVER)
    try:
        s.options["max_cpu_time"] = float(SOLVER_TMAX_S)
        s.options["tol"] = 1e-6
        s.options["constr_viol_tol"] = 1e-6
    except Exception:
        pass
    return s

def subclass_num(sid: str):
    m = re.search(r'(\d+)$', str(sid))
    return int(m.group(1)) if m else None

# -----------------------
# LOAD INPUTS
# -----------------------
xls = pd.ExcelFile(excel_path, engine="openpyxl")
arch = pd.read_excel(xls, sheet_name="archetypes")
db   = pd.read_excel(xls, sheet_name="nrel_db")

# Normalize numeric for computation (in-memory only)
arch["_Emin"] = arch["out.electricity.total.energy_consumption_min"].apply(to_num)
arch["_Emax"] = arch["out.electricity.total.energy_consumption_max"].apply(to_num)
arch["_H"]    = arch["HousingStock"].apply(to_num)
arch["_TotE"] = arch["annual_consumption_kwh"].apply(to_num)

db["_E"]   = pd.to_numeric(db["out.electricity.total.energy_consumption"].apply(to_num), errors="coerce")
db["_bed"] = pd.to_numeric(db["in.bedrooms"].apply(to_num), errors="coerce")

# -----------------------
# SELECT SUBCLASS RANGE
# -----------------------
from_n = subclass_num(FROM_SUBCLASS)
to_n   = subclass_num(TO_SUBCLASS)
if from_n is None or to_n is None:
    raise ValueError(f"Could not parse subclass numbers from: {FROM_SUBCLASS}, {TO_SUBCLASS}")
if from_n > to_n:
    from_n, to_n = to_n, from_n

arch["_sid_num"] = arch["sub_class_id"].astype(str).map(subclass_num)
sel_mask = arch["_sid_num"].between(from_n, to_n, inclusive="both")
arch_sel = arch[sel_mask].copy().reset_index(drop=True)

if arch_sel.empty:
    raise RuntimeError(f"No archetypes found in range Subclass_{from_n:02d}..Subclass_{to_n:02d}")

print(f"Running subclasses (energy-only): {arch_sel['sub_class_id'].tolist()}")

# -----------------------
# PREP OUTPUT FRAMES
# -----------------------
all_bldg_ids = db["bldg_id"].tolist()
weights_out  = pd.DataFrame({"bldg_id": all_bldg_ids})
summary_rows = []

solver = get_solver()

# -----------------------
# MAIN LOOP (ENERGY-ONLY)
# -----------------------
for _, arow in arch_sel.iterrows():
    sid = str(arow["sub_class_id"])
    t0 = time.perf_counter()

    H = int(round(arow["_H"])) if np.isfinite(arow["_H"]) else 0
    TotE = float(arow["_TotE"]) if np.isfinite(arow["_TotE"]) else np.nan

    if H <= 0:
        weights_out[sid] = 0.0
        summary_rows.append({
            "sub_class_id": sid, "status": "skipped_H<=0",
            "solve_sec": 0.0,
            "candidates_N": 0,
            "H_target": H, "H_lo": np.nan, "H_hi": np.nan, "sum_w": 0.0,
            "E_min": float(arow["_Emin"]), "E_max": float(arow["_Emax"]),
            "TotalEnergy_target_kwh": TotE,
            "TotalEnergy_actual_kwh": 0.0,
            "energy_rel_error_pct": np.nan,
            "rho_used": np.nan, "L_min": np.nan, "L_max": np.nan,
            "U_min": np.nan, "U_max": np.nan,
            "objective_value": np.nan, "obj_energy": np.nan, "obj_reg": np.nan,
            "active_archetypes_w>0": 0, "top1_share": np.nan, "top5_share": np.nan,
            "N_eff": 0.0, "gini": 0.0,
            "bt_filter": str(arow.get("in.geometry_building_type_recs") or "any"),
            "bedrooms_filter": str(arow.get("in.bedrooms") or "any"),
            "heating_fuel_filter": str(arow.get("in.heating_fuel") or "any"),
        })
        continue

    E_min = float(arow["_Emin"]) if np.isfinite(arow["_Emin"]) else -np.inf
    E_max = float(arow["_Emax"]) if np.isfinite(arow["_Emax"]) else  np.inf
    if E_min >= E_max:
        E_min, E_max = -np.inf, np.inf

    # ---- Candidate filters ----
    cand = db[(db["_E"] >= E_min) & (db["_E"] < E_max)].copy()

    bt_list = parse_building_type_filter(arow.get("in.geometry_building_type_recs"))
    if bt_list:
        bt_lower = {b.lower() for b in bt_list}
        cand = cand[cand["in.geometry_building_type_recs"].astype(str).str.strip().str.lower().isin(bt_lower)]

    mask_bed = bedrooms_mask(cand["_bed"], arow.get("in.bedrooms"))
    cand = cand[mask_bed]

    hf = arow.get("in.heating_fuel")
    if isinstance(hf, str) and hf.strip().lower() != "any" and hf.strip() != "":
        cand = cand[cand["in.heating_fuel"].astype(str).str.strip().str.lower() == hf.strip().lower()]

    N = len(cand)
    if N == 0:
        weights_out[sid] = 0.0
        summary_rows.append({
            "sub_class_id": sid, "status": "no_candidates_after_filter",
            "solve_sec": 0.0,
            "candidates_N": 0,
            "H_target": H,
            "H_lo": np.nan, "H_hi": np.nan, "sum_w": 0.0,
            "E_min": E_min, "E_max": E_max,
            "TotalEnergy_target_kwh": TotE,
            "TotalEnergy_actual_kwh": 0.0,
            "energy_rel_error_pct": np.nan,
            "rho_used": np.nan, "L_min": np.nan, "L_max": np.nan,
            "U_min": np.nan, "U_max": np.nan,
            "objective_value": np.nan, "obj_energy": np.nan, "obj_reg": np.nan,
            "active_archetypes_w>0": 0, "top1_share": np.nan, "top5_share": np.nan,
            "N_eff": 0.0, "gini": 0.0,
            "bt_filter": "; ".join(bt_list) if bt_list else "any",
            "bedrooms_filter": str(arow.get("in.bedrooms")),
            "heating_fuel_filter": str(arow.get("in.heating_fuel")),
        })
        continue

    # Data vectors
    E = cand["_E"].to_numpy(dtype=float)

    # Bounds
    rho = auto_shrink_rho_for_feasibility(float(rho_min_weight), H, N)
    L = rho * H / max(N, 1) * np.ones(N, dtype=float)
    U = np.full(N, wmax_fraction_of_H * H, dtype=float) if (wmax_fraction_of_H is not None) else np.full(N, float('inf'))

    # Sum band
    H_lo = (1.0 - float(sum_band_eps)) * float(H)
    H_hi = (1.0 + float(sum_band_eps)) * float(H)

    # Energy per HH target
    E_star = (TotE / H) if (np.isfinite(TotE) and H > 0) else np.nan

    # ---- Build & solve energy-only model ----
    model = ConcreteModel()
    model.K = PyomoSet(initialize=list(range(N)))

    def w_bounds(m, k):
        return (float(L[k]), float(U[k]))
    model.w = Var(model.K, domain=NonNegativeReals, bounds=w_bounds)

    if sum_band_eps <= 0.0:
        def sum_w_eq(m): return sum(m.w[k] for k in m.K) == float(H)
        model.sum_w_eq = Constraint(rule=sum_w_eq)
    else:
        model.sum_w_lo = Constraint(expr=sum(model.w[k] for k in model.K) >= H_lo)
        model.sum_w_hi = Constraint(expr=sum(model.w[k] for k in model.K) <= H_hi)

    def objective_rule(m):
        energy_target = float(H) * (0.0 if not np.isfinite(E_star) else float(E_star))
        energy_dev = (sum(m.w[k] * E[k] for k in m.K) - energy_target)**2
        reg = sum(m.w[k]**2 for k in m.K)
        return float(alpha)*energy_dev + float(gamma)*reg
    model.obj = Objective(rule=objective_rule)

    try:
        res = solver.solve(model, tee=False)
        solver_status = str(res.solver.status)
        solver_term   = str(res.solver.termination_condition)
    except Exception as e:
        solver_status = "exception"
        solver_term   = f"{e}"

    # Extract & map
    w_opt = np.array([value(model.w[k]) if model.w[k].value is not None else 0.0 for k in model.K])
    col = np.zeros(len(db), dtype=float)
    col[cand.index.to_numpy()] = w_opt
    weights_out[sid] = col

    # Diagnostics
    sum_w = float(np.nansum(w_opt))
    active = int(np.sum(w_opt > 0))
    TotalEnergy_actual = float(np.dot(w_opt, E))
    energy_rel_error = ((TotalEnergy_actual - TotE) / TotE * 100.0) if (np.isfinite(TotE) and TotE != 0) else np.nan

    _E_star_val = 0.0 if not np.isfinite(E_star) else float(E_star)
    obj_energy = float((np.dot(w_opt, E) - float(H)*_E_star_val)**2)
    obj_reg    = float(np.sum(w_opt**2))
    obj_total  = float(alpha*obj_energy + gamma*obj_reg)

    if N > 0 and sum_w > 0:
        ws = np.sort(w_opt)[::-1]
        top1_share = float(ws[0] / sum_w)
        top5_share = float(ws[:min(5, N)].sum() / sum_w)
    else:
        top1_share = top5_share = np.nan

    t1 = time.perf_counter()

    summary_rows.append({
        "sub_class_id": sid,
        "status": f"{solver_status}/{solver_term}",
        "solve_sec": t1 - t0,
        "candidates_N": int(N),

        "H_target": int(H),
        "H_lo": H_lo, "H_hi": H_hi, "sum_w": sum_w,

        "E_min": E_min, "E_max": E_max,
        "TotalEnergy_target_kwh": float(TotE) if np.isfinite(TotE) else np.nan,
        "TotalEnergy_actual_kwh": float(TotalEnergy_actual),
        "energy_rel_error_pct": float(energy_rel_error) if np.isfinite(energy_rel_error) else np.nan,

        "rho_used": float(rho),
        "L_min": float(np.min(L)), "L_max": float(np.max(L)),
        "U_min": float(np.min(U)), "U_max": float(np.max(U)),

        "objective_value": obj_total,
        "obj_energy": obj_energy,
        "obj_reg": obj_reg,

        "active_archetypes_w>0": active,
        "top1_share": top1_share,
        "top5_share": top5_share,
        "N_eff": effective_sample_size(w_opt),
        "gini": gini_coef(w_opt),

        "bt_filter": "; ".join(bt_list) if bt_list else "any",
        "bedrooms_filter": str(arow.get("in.bedrooms")),
        "heating_fuel_filter": str(arow.get("in.heating_fuel")),
    })

# -----------------------
# SAVE TO NEW FILE
# -----------------------
stem = Path(excel_path).with_suffix("").name
default_name = f"{stem}__weights_{FROM_SUBCLASS}_to_{TO_SUBCLASS}__energy_only.xlsx"
out_path = Path(excel_path).with_name(save_as or default_name)

with pd.ExcelWriter(out_path, engine="openpyxl", mode="w") as writer:
    weights_out.to_excel(writer, sheet_name="weights", index=False)
    pd.DataFrame(summary_rows).to_excel(writer, sheet_name="weights_summary", index=False)

print(f"Done. Wrote energy-only range results to: {out_path}")


Running subclasses (energy-only): ['Subclass_01', 'Subclass_02', 'Subclass_03', 'Subclass_04', 'Subclass_05', 'Subclass_06', 'Subclass_07', 'Subclass_08', 'Subclass_09', 'Subclass_10', 'Subclass_11', 'Subclass_12', 'Subclass_13', 'Subclass_14', 'Subclass_15', 'Subclass_16', 'Subclass_17', 'Subclass_18', 'Subclass_19']
Done. Wrote energy-only range results to: C:\Users\dmitr\Article_02\nrel_weights\weight_solver_13__weights_Subclass_01_to_Subclass_19__energy_only.xlsx


%%markdown
# **Optimize a Range of Subclasses (Energy-Only + Optional Appliance Bands) and Save to New File**  
*(Mathematical/algorithmic spec mirroring the Jupyter cell you shared; same notation/style as prior specs.)*

## Sets and indices
- $\mathcal{B}$: full set of **building (archetype) rows** in the NREL DB (column `bldg_id`).  
- For a given subclass $s$ in the selected range, let $\mathcal{K}(s)\subseteq\mathcal{B}$ be the **candidate set** after filters (energy band, building-type, bedrooms, heating fuel).  
- Index $k\in\mathcal{K}(s)$ enumerates candidate buildings for subclass $s$.

---

## Inputs (from workbook)
For each subclass row $s$ in `archetypes` (only those with `select_weights == "yes"` are optimized; others are **skipped** but still appear in outputs with zeros):
- Energy band: $E^{\min}(s)$, $E^{\max}(s)$ from columns `out.electricity.total.energy_consumption_min/max`.
- Housing stock target: $H(s)$ from `HousingStock` (integer target mass of households to allocate).
- Annual total energy target (optional): $\mathrm{TotE}(s)$ from `annual_consumption_kwh`.
- Filters (optional):  
  `in.geometry_building_type_recs` $\Rightarrow$ a set of allowed building types (or **any**),  
  `in.bedrooms` $\Rightarrow$ allowed counts (or **any**),  
  `in.heating_fuel` $\Rightarrow$ a single allowed fuel (or **any**).
- Toggle for appliance penetrations: column **`applianc_ penetrations`** $\in\{\text{yes},\text{no}\}$ (note the space in the column name).

From `nrel_db` (the candidate table), for each building $k$:
- Hour-agnostic energy per household (kWh/yr): $E_k=$ `out.electricity.total.energy_consumption`.
- Feature flags for appliances (0/1): for each field $f$ listed in `APPLIANCE_CONSTRAINTS`.

Global/user config:
- Objective weights: $\alpha>0$ (energy deviation), $\gamma\ge 0$ (L2 on $w$).
- Sum-band half-width $\varepsilon\in[0,1)$ (e.g., $0.03$ for $\pm 3\%$).
- Lower envelope parameter $\rho>0$ for per-candidate floors (auto-shrunk if needed).
- Optional per-candidate **upper** cap as a fraction of $H(s)$: $\omega\in(0,1]$ or disabled.
- Solver: IPOPT with tolerances/max time.

---

## Candidate pre-filtering (per subclass $s$)
Let $E_k$ be numeric and finite; keep $k$ with
- **Energy band**: $E^{\min}(s)\le E_k < E^{\max}(s)$ (if the band is invalid or missing, treat as $(-\infty,\infty)$).
- **Building type**: $k$ passes if its `in.geometry_building_type_recs` is in the allowed set (case-insensitive), else drop; **any** keeps all.
- **Bedrooms**: $k$ passes if integer bedroom count is in the allowed set; **any** keeps all.
- **Heating fuel**: either equal to the requested fuel (case-insensitive) or **any**.

If no candidates remain ($\mathcal{K}(s)=\varnothing$), subclass is reported as **no\_candidates\_after\_filter** and all weights for $s$ are zeros.

---

## Decision variables (per subclass $s$)
For each $k\in\mathcal{K}(s)$, choose a **weight** $w_k\ge 0$ (mass of households assigned to candidate $k$).

> Interpretation: $\sum_{k\in\mathcal{K}(s)} w_k$ approximates the subclass housing stock $H(s)$ up to a small tolerance band; $\sum_k w_k\,E_k$ approximates the subclass total energy target.

---

## Bounds and envelopes (per subclass $s$)
Let $N=|\mathcal{K}(s)|$ and $H=H(s)$. Define:
- **Auto-shrunken floor** (to guarantee feasibility):  
  $$\rho^\star=\min\!\left(\rho,\ \frac{0.99\,H}{H+\mathbf{1}[H=0]}\right),\qquad
  L_k=\frac{\rho^\star\,H}{\max(N,1)}\ \ \forall k.$$
  (If $\rho H/N>H$ the routine shrinks $\rho$ just enough so $\sum_k L_k\le H$.)
- **Upper caps** (optional):  
  $$U_k=\begin{cases}
  \omega\,H,& \text{if }\omega\ \text{is provided},\\
  +\infty,& \text{otherwise.}
  \end{cases}$$
Thus the **box** is $L_k\le w_k\le U_k$ for all $k$.

---

## Sum (mass) band
Let
\[
H_{\mathrm{lo}}=(1-\varepsilon)\,H,\qquad H_{\mathrm{hi}}=(1+\varepsilon)\,H.
\]
- If $\varepsilon=0$: enforce the **equality** $\sum_k w_k=H$.
- If $\varepsilon>0$: enforce the **band**
\[
H_{\mathrm{lo}}\ \le\ \sum_{k\in\mathcal{K}(s)} w_k\ \le\ H_{\mathrm{hi}}.
\]

---

## Energy target and objective
Define the **per-household** energy target (if available):
\[
E^\star=\begin{cases}
\mathrm{TotE}(s)/H,& \text{if } H>0 \text{ and TotE is finite},\\
\text{undefined (ignored)},& \text{otherwise}.
\end{cases}
\]
The **energy deviation** term is
\[
\bigg(\sum_{k} w_k\,E_k - H\,E^\star\bigg)^2
\quad\text{(interpreted as }0\text{ if }E^\star\text{ undefined).}
\]
Add an L2 regularizer on the weights:
\[
\sum_{k} w_k^2.
\]
**Objective (convex QP):**
\[
\min_{w}\ \ \alpha\ \Big(\sum_{k} w_k\,E_k - H\,E^\star\Big)^2\ +\ \gamma\ \sum_{k} w_k^2.
\]

---

## Optional: appliance penetration bands (linear)
Enabled **only if** the subclass row has **`applianc_ penetrations == "yes"`**, and **only for** constraint fields $f$ with `active: True` in `APPLIANCE_CONSTRAINTS`.

For a field $f$ with target fraction $t\in[0,1]$ and absolute tolerance $d\ge0$:
\[
\ell=\max(0, t-d),\qquad u=\min(1, t+d),
\]
let $A_k^{(f)}\in\{0,1\}$ be the presence flag (coerced from data; missing $\Rightarrow 0$).  
With $W=\sum_k w_k$, enforce
\[
\ell\,W \ \le\ \sum_{k} w_k\,A_k^{(f)}\ \le\ u\,W.
\]
This keeps the **share** of the selected sample within $[\,\ell,\ u\,]$.

---

## Full model (per subclass $s$)
\[
\begin{aligned}
\min_{w}\quad & \alpha\left(\sum_{k} w_k E_k - H E^\star\right)^2 + \gamma\sum_{k} w_k^2\\[4pt]
\text{s.t.}\quad
& L_k \le w_k \le U_k,\quad \forall k\in\mathcal{K}(s),\\
& \sum_{k} w_k = H && (\varepsilon=0),\\
& H_{\mathrm{lo}} \le \sum_{k} w_k \le H_{\mathrm{hi}} && (\varepsilon>0),\\
& \ell_f \sum_k w_k \le \sum_k w_k A_k^{(f)} \le u_f \sum_k w_k && \text{for each active appliance } f \text{ (only if enabled).}
\end{aligned}
\]

---

## Solver and feasibility handling
- Solved with IPOPT (convex QP here due to squared terms and linear constraints).  
- If the solver does **not** return an (locally) optimal solution: set $w_k\leftarrow 0$ for all $k$ and mark the status as infeasible/failed.

---

## Execution over a range and selection gate
- Let the inclusive subclass range be `Subclass_m` … `Subclass_n`.  
- For each $s$ in this range:  
  - If `select_weights != "yes"`: **skip** optimization; keep the column of $s$ **all zeros** and record status `skipped_select_weights=no`.  
  - Else, run the model above on $\mathcal{K}(s)$ constructed from the filters.

---

## Outputs (three Excel sheets; all subclasses in range included)
1. **`weights`** (wide, by `bldg_id`):  
   For every subclass column $s$ in the range, writes the optimized vector $\{w_k\}$; skipped/empty cases remain **zeros**.
2. **`weights_normalized`** (wide, by `bldg_id`):  
   Column-wise normalization: for each subclass $s$ with $\sum_k w_k>0$, set $\tilde w_k = w_k/\sum_k w_k$; otherwise keep zeros.
3. **`weights_summary`** (one row per subclass $s$): includes
   - `status` (solver status / skip reason), `solve_sec`, `candidates_N`.  
   - $H$, $H_{\mathrm{lo}}$, $H_{\mathrm{hi}}$, $\sum_k w_k$, and `housingstock_error_pct` $=100\cdot(\sum w - H)/H$.  
   - Energy band $[E^{\min},E^{\max})$, $\mathrm{TotE}$, realized $\sum_k w_k E_k$, and relative error (if $\mathrm{TotE}$ is finite).  
   - $\rho$ used, $\min/\max$ of $L_k$ and $U_k$.  
   - Objective breakdown: total value, energy term, regularizer.  
   - Diversity stats: number of positive weights, top-1 share, top-5 share, **effective sample size** $N_{\mathrm{eff}}=\dfrac{(\sum_k w_k)^2}{\sum_k w_k^2}$, **Gini** over $\{w_k\}$.  
   - Echoed filters: building-type, bedrooms, heating fuel; whether appliance constraints were applied.

---

## File naming
If `save_as` is `None`, the output file name is auto-formed as  
\[
\texttt{\<stem\>\_\_weights\_Subclass\_m\_to\_Subclass\_n\_\_energy\_appliance.xlsx},
\]
where `<stem>` is the source workbook stem.

---

## Edge cases & notes
- If $H\le 0$: skip optimization for $s$ and keep zeros; report `skipped_H<=0`.  
- If $\mathcal{K}(s)=\varnothing$: keep zeros; report `no_candidates_after_filter`.  
- Appliance constraints require both: (a) subclass toggle **yes**, and (b) per-field `active: True`. Missing appliance columns are treated as zeros.  
- All numeric parsing coerces commas/strings to floats; non-parsable values $\rightarrow \mathrm{NaN}$ and then handled as described above.


%%markdown
# Iterative Calibration per Archetype (Subclass) — Constraint–Loosening Protocol and Outcomes

This note documents **how we modeled, by iterations, different subclasses (archetypes)** using the same QP weighting model and **gradually loosening appliance-penetration constraints** only as needed to reach feasibility and a good energy fit. It mirrors the notation of the previous spec.

---

## Global model & invariants (shared across all subclasses)
Solver/config (unchanged unless stated):
- `SOLVER="ipopt"`, `IPOPT_EXE=""`, `SOLVER_TMAX_S=60`.
- Objective weights: $\alpha=1.0$ (energy deviation squared), $\gamma=10^{-5}$ (L2 on $w$).
- Sum band: $\varepsilon=0.03$ (i.e., $H_{\mathrm{lo}}=(1-\varepsilon)H$, $H_{\mathrm{hi}}=(1+\varepsilon)H$).
- Per-candidate lower floor: $L_k=\rho^\star H/\max(N,1)$ with $\rho_{\min}=0.0010$ and auto-shrink; upper cap disabled: `wmax_fraction_of_H=None`.
- Appliance constraint format (when active):
  \[
  \ell_f=\max(0, t_f-d_f),\quad u_f=\min(1, t_f+d_f),\qquad
  \ell_f\!\sum_k w_k \le \sum_k w_k A_k^{(f)} \le u_f\!\sum_k w_k.
  \]
  Targets: $t_{\text{hvac}}=0.20,\ t_{\text{washer}}=0.92,\ t_{\text{freezer}}=0.12,\ t_{\text{dish}}=0.10$.
- Baseline (“tight”) tolerance when active: $d_f=0.005$ unless stated otherwise.

---

## Iterative scheme (constraint–loosening ladder)
For each subclass $s$ we solved the convex QP and **progressed through the ladder only if** the current level was **infeasible** or produced an unacceptable data fit (e.g., solver failure or severe energy/stock deviations). The ladder:

- **L0 — Tight, all active (baseline).**  
  Active: hvac, washer, freezer, dishwasher. Tolerance $d_f=0.005$ for all.

- **L1 — Widen tolerance (moderate).**  
  Keep the same active set, but increase $d_f$ to $0.05$ for selected fields (as specified below per subclass).

- **L2 — Deactivate selected fields.**  
  Turn off one or more appliance constraints (keep others per subclass note).

- **L3 — Liberalize deactivated fields further (if needed).**  
  Where a field remains informative but erratic, either leave it **inactive** or (rare case) allow a very wide tolerance (documented below).

> We stop as soon as a level yields an **optimal or locally optimal** solution with acceptable diagnostics (mass within band, reasonable energy error, stable weights).

---

## Outcomes by subclass (final level & active constraints)

### A. Subclasses that solved at **L0 (tight; all active; $d_f=0.005$)**
**Subclass_01, Subclass_03, Subclass_05, Subclass_09, Subclass_10, Subclass_13, Subclass_17, Subclass_18**

- **Final level:** L0  
- **Active constraints:** hvac (±0.005), washer (±0.005), freezer (±0.005), dishwasher (±0.005)  
- **Rationale:** Candidate pool supported tight appliance shares without violating the mass band or energy target.

---

### B. Subclass_19 — solved at **L1 (moderate widening)**
- **Final level:** L1  
- **Active constraints:** hvac (±0.05), washer (±0.05), freezer (±0.05), dishwasher (±0.05)  
- **Rationale:** Tight bands caused infeasibility/strain; modest widening restored feasibility with stable weights.

---

### C. Subclass_02 — solved at **L2 (deactivate HVAC only)**
- **Final level:** L2  
- **Active constraints:** washer (±0.005), freezer (±0.005), dishwasher (±0.005)  
- **Inactive:** hvac  
- **Rationale:** HVAC flag distribution conflicted with other filters; turning it off allowed a solution at tight bands for the remaining fields.

---

### D. Subclass_11 — solved at **L2 (all appliance constraints off)**
- **Final level:** L2  
- **Active constraints:** *none* (hvac, washer, freezer, dishwasher all inactive)  
- **Rationale:** Severe mismatch/coverage issues across all appliance flags; appliance guidance removed to prioritize mass/energy fit.

---

### E. Subclass_06, Subclass_07 — solved at **L3 (mixed: core on, others off/wide)**
- **Final level:** L3  
- **Active constraints:** hvac (±0.05), washer (±0.05)  
- **Inactive / extremely liberalized:** freezer (inactive; tol shown as 0.5 for documentation), dishwasher (inactive; tol 0.5)  
- **Rationale:** Core end-uses (HVAC, washer) remained informative with moderate tolerance; freezer/dishwasher flags were sparse/noisy → deactivated.

---

### F. Subclass_04, Subclass_08, Subclass_16 — solved at **L1 (moderate widening; washer off)**
- **Final level:** L1/L2 hybrid (per spec below)  
- **Active constraints:** hvac (±0.05), freezer (±0.05), dishwasher (±0.05)  
- **Inactive:** washer  
- **Rationale:** Washer share was incompatible with available candidates after filters; other three constraints worked with moderate bands.

---

### G. Subclass_12 — solved at **L1/L2 (hvac on; others off)**
- **Final level:** L2  
- **Active constraints:** hvac (±0.05)  
- **Inactive:** washer, freezer, dishwasher  
- **Rationale:** Only HVAC remained predictive/feasible; others removed to stabilize the solution.

---

### H. Subclass_14, Subclass_15 — solved at **L2/L3 (all off; hvac documented wide)**
- **Final level:** L2  
- **Active constraints:** *none* (hvac, washer, freezer, dishwasher all inactive)  
- **Note:** hvac recorded with a very wide nominal tol (±0.3) to reflect data permissiveness, but effectively treated as **inactive**.  
- **Rationale:** Appliance shares not reliable under the subclass filters; best results obtained by relying on energy/mass only.

---

## Summary table (final appliance settings per subclass)

| Subclass     | Final Level | HVAC            | Washer          | Freezer         | Dishwasher      |
|--------------|-------------|-----------------|-----------------|-----------------|-----------------|
| 01,03,05,09,10,13,17,18 | L0          | active ±0.005  | active ±0.005  | active ±0.005  | active ±0.005  |
| 19           | L1          | active ±0.05   | active ±0.05   | active ±0.05   | active ±0.05   |
| 02           | L2          | **inactive**    | active ±0.005  | active ±0.005  | active ±0.005  |
| 11           | L2          | **inactive**    | **inactive**    | **inactive**    | **inactive**    |
| 06,07        | L3          | active ±0.05   | active ±0.05   | **inactive** *(±0.5 doc)* | **inactive** *(±0.5 doc)* |
| 04,08,16     | L1/L2       | active ±0.05   | **inactive**    | active ±0.05   | active ±0.05   |
| 12


In [36]:
# =========================
# Cell 2 — OPTIMIZE A RANGE OF SUBCLASSES (ENERGY-ONLY + OPTIONAL APPLIANCE BANDS) AND SAVE TO NEW FILE
# =========================
# - Runs energy matching per subclass only when archetypes.select_weights == "yes"
# - Appliance bands added only when archetypes['applianc_ penetrations'] == "yes"
# - Uses softened sum constraint: (1-ε)H <= Σw <= (1+ε)H
# - Writes 'weights', 'weights_normalized', and 'weights_summary' to a NEW Excel file
#
# Assumes validation cell passed. This cell does not modify inputs.

import re
import time
import numpy as np
import pandas as pd
from pathlib import Path
from pyomo.environ import (
    ConcreteModel, Var, Objective, Constraint, NonNegativeReals, SolverFactory, value,
    Set as PyomoSet
)

# -----------------------
# USER CONFIG
# -----------------------
excel_path = r"C:\Users\dmitr\Article_02\nrel_weights/weight_solver_15i.xlsx"  # source workbook
FROM_SUBCLASS = "Subclass_01"   # inclusive
TO_SUBCLASS   = "Subclass_19"   # inclusive

# If None, auto-names "<stem>__weights_<FROM>_to_<TO>__energy_appliance.xlsx"
save_as = None

# Solver config
SOLVER        = "ipopt"
IPOPT_EXE     = ""   # set a path if ipopt.exe is not on PATH
SOLVER_TMAX_S = 60

# Objective weights
alpha = 1.0     # energy squared deviation
gamma = 1e-5    # L2 regularization on w

# Sum band (soften Σw=H)
sum_band_eps = 0.03   # ±3%

# Weight bounds
rho_min_weight         = 0.0010   # floor: L_k = rho * H/N (auto-shrunk if needed)
wmax_fraction_of_H     = None     # e.g., 0.15 to cap per-candidate; None to disable

# -----------------------
# APPLIANCE CONSTRAINTS (per-field tol_abs)
# -----------------------
# Applied only when archetypes row has column 'applianc_ penetrations' == "yes" (case-insensitive)
APPLIANCE_CONSTRAINTS = {
    "in.hvac_cooling_type":       {"active": False, "target_pct": 0.20, "tol_abs": 0.3},
    "in.clothes_washer_presence": {"active": False, "target_pct": 0.92, "tol_abs": 0.05},
    "in.misc_freezer":            {"active": False, "target_pct": 0.12, "tol_abs": 0.05},
    "in.dishwasher":              {"active": False, "target_pct": 0.10, "tol_abs": 0.05},
}
# -----------------------
# HELPERS
# -----------------------
def to_num(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x)
    s = str(x).replace(",", "").strip()
    if s == "" or s.lower() in {"none", "nan"}:
        return np.nan
    try:
        return float(s)
    except Exception:
        return np.nan

def parse_building_type_filter(txt):
    """Allow 'any' or multiple options separated by 'or' (quotes optional)."""
    if txt is None:
        return None
    s = str(txt).strip()
    if s == "" or s.lower() == "any":
        return None
    parts = [p.strip().strip('"').strip("'") for p in re.split(r'\s+or\s+', s, flags=re.I)]
    parts = [p for p in parts if p]
    return parts or None

def bedrooms_mask(series, token):
    """Accepts: any / blank / '3' / '3 or 4 or 5' / '3 or 4 bedrooms'."""
    if token is None:
        return pd.Series(True, index=series.index)
    t = str(token).strip().lower()
    if t == "" or t == "any":
        return pd.Series(True, index=series.index)
    nums = [int(x) for x in re.findall(r'\d+', t)]
    if nums:
        want = set(nums)
        return series.round().astype("Int64").isin(list(want)).fillna(False)
    return pd.Series(True, index=series.index)

def auto_shrink_rho_for_feasibility(rho, H, N):
    sumL = rho * H / max(N, 1) * N
    if sumL <= H:
        return rho
    # shrink just enough to be feasible
    return (0.99 * H) / max(H, 1e-12)

def effective_sample_size(w):
    w = np.asarray(w, dtype=float)
    s = w.sum()
    if s <= 0:
        return 0.0
    return float((s * s) / (np.sum(w * w) + 1e-12))

def gini_coef(w):
    w = np.asarray(w, dtype=float)
    if w.size == 0 or np.allclose(w, 0):
        return 0.0
    sw = np.sort(w)
    n = w.size
    cum = np.cumsum(sw)
    return float((n + 1 - 2 * np.sum(cum) / cum[-1]) / n)

def get_solver():
    s = SolverFactory(SOLVER, executable=IPOPT_EXE) if (IPOPT_EXE and Path(IPOPT_EXE).exists()) else SolverFactory(SOLVER)
    try:
        s.options["max_cpu_time"] = float(SOLVER_TMAX_S)
        s.options["tol"] = 1e-6
        s.options["constr_viol_tol"] = 1e-6
    except Exception:
        pass
    return s

def subclass_num(sid: str):
    m = re.search(r'(\d+)$', str(sid))
    return int(m.group(1)) if m else None

def is_yes(x: object) -> bool:
    if pd.isna(x):
        return False
    return str(x).strip().lower() == "yes"

# -----------------------
# LOAD INPUTS
# -----------------------
xls  = pd.ExcelFile(excel_path, engine="openpyxl")
arch = pd.read_excel(xls, sheet_name="archetypes")
db   = pd.read_excel(xls, sheet_name="nrel_db")

# Normalize numeric for computation (in-memory only)
arch["_Emin"] = arch["out.electricity.total.energy_consumption_min"].apply(to_num)
arch["_Emax"] = arch["out.electricity.total.energy_consumption_max"].apply(to_num)
arch["_H"]    = arch["HousingStock"].apply(to_num)
arch["_TotE"] = arch["annual_consumption_kwh"].apply(to_num)

db["_E"]   = pd.to_numeric(db["out.electricity.total.energy_consumption"].apply(to_num), errors="coerce")
db["_bed"] = pd.to_numeric(db["in.bedrooms"].apply(to_num), errors="coerce")

# For appliance flags: coerce to {0,1}, NaN -> 0, nonzero -> 1
for f in APPLIANCE_CONSTRAINTS.keys():
    if f in db.columns:
        col = pd.to_numeric(db[f], errors="coerce").fillna(0.0)
        db[f] = (col != 0).astype(int)
    else:
        # If field absent, create as zeros (no presence)
        db[f] = 0

# -----------------------
# SELECT SUBCLASS RANGE
# -----------------------
from_n = subclass_num(FROM_SUBCLASS)
to_n   = subclass_num(TO_SUBCLASS)
if from_n is None or to_n is None:
    raise ValueError(f"Could not parse subclass numbers from: {FROM_SUBCLASS}, {TO_SUBCLASS}")
if from_n > to_n:
    from_n, to_n = to_n, from_n

arch["_sid_num"] = arch["sub_class_id"].astype(str).map(subclass_num)
sel_mask = arch["_sid_num"].between(from_n, to_n, inclusive="both")
arch_sel = arch[sel_mask].copy().reset_index(drop=True)
if arch_sel.empty:
    raise RuntimeError(f"No archetypes found in range Subclass_{from_n:02d}..Subclass_{to_n:02d}")

print(f"Running subclasses (full list for output): {arch_sel['sub_class_id'].tolist()}")

# -----------------------
# PREP OUTPUT FRAMES (include ALL subclasses in range)
# -----------------------
all_bldg_ids = db["bldg_id"].tolist()
weights_out  = pd.DataFrame({"bldg_id": all_bldg_ids})
# Pre-create zero columns for every subclass in the range
for sid_all in arch_sel["sub_class_id"].astype(str):
    weights_out[sid_all] = 0.0

summary_rows = []
solver = get_solver()

# -----------------------
# MAIN LOOP
# -----------------------
APPLIANCE_FLAG_COL = "applianc_ penetrations"  # note: space per your sheet

for _, arow in arch_sel.iterrows():
    sid = str(arow["sub_class_id"])
    t0  = time.perf_counter()

    # Selection gate
    if not is_yes(arow.get("select_weights")):
        # keep zeros in weights_out[sid]
        summary_rows.append({
            "sub_class_id": sid,
            "status": "skipped_select_weights=no",
            "solve_sec": 0.0,
            "candidates_N": 0,
            "H_target": int(round(arow["_H"])) if np.isfinite(arow["_H"]) else 0,
            "H_lo": np.nan, "H_hi": np.nan,
            "sum_w": 0.0,
            "housingstock_error_pct": np.nan,
            "E_min": float(arow["_Emin"]) if np.isfinite(arow["_Emin"]) else np.nan,
            "E_max": float(arow["_Emax"]) if np.isfinite(arow["_Emax"]) else np.nan,
            "TotalEnergy_target_kwh": float(arow["_TotE"]) if np.isfinite(arow["_TotE"]) else np.nan,
            "TotalEnergy_actual_kwh": 0.0,
            "energy_rel_error_pct": np.nan,
            "rho_used": np.nan,
            "L_min": np.nan, "L_max": np.nan,
            "U_min": np.nan, "U_max": np.nan,
            "objective_value": np.nan,
            "obj_energy": np.nan, "obj_reg": np.nan,
            "active_archetypes_w>0": 0,
            "top1_share": np.nan, "top5_share": np.nan,
            "N_eff": 0.0, "gini": 0.0,
            "bt_filter": str(arow.get("in.geometry_building_type_recs") or "any"),
            "bedrooms_filter": str(arow.get("in.bedrooms") or "any"),
            "heating_fuel_filter": str(arow.get("in.heating_fuel") or "any"),
            "appliance_constraints_applied": False,
        })
        continue

    H    = int(round(arow["_H"])) if np.isfinite(arow["_H"]) else 0
    TotE = float(arow["_TotE"])   if np.isfinite(arow["_TotE"]) else np.nan

    # Short-circuit if no HousingStock
    if H <= 0:
        # weights_out[sid] already zeros
        summary_rows.append({
            "sub_class_id": sid,
            "status": "skipped_H<=0",
            "solve_sec": 0.0,
            "candidates_N": 0,
            "H_target": H,
            "H_lo": np.nan, "H_hi": np.nan,
            "sum_w": 0.0,
            "housingstock_error_pct": np.nan,
            "E_min": float(arow["_Emin"]),
            "E_max": float(arow["_Emax"]),
            "TotalEnergy_target_kwh": TotE,
            "TotalEnergy_actual_kwh": 0.0,
            "energy_rel_error_pct": np.nan,
            "rho_used": np.nan,
            "L_min": np.nan, "L_max": np.nan,
            "U_min": np.nan, "U_max": np.nan,
            "objective_value": np.nan,
            "obj_energy": np.nan, "obj_reg": np.nan,
            "active_archetypes_w>0": 0,
            "top1_share": np.nan, "top5_share": np.nan,
            "N_eff": 0.0, "gini": 0.0,
            "bt_filter": str(arow.get("in.geometry_building_type_recs") or "any"),
            "bedrooms_filter": str(arow.get("in.bedrooms") or "any"),
            "heating_fuel_filter": str(arow.get("in.heating_fuel") or "any"),
            "appliance_constraints_applied": False,
        })
        continue

    # Candidate filters
    E_min = float(arow["_Emin"]) if np.isfinite(arow["_Emin"]) else -np.inf
    E_max = float(arow["_Emax"]) if np.isfinite(arow["_Emax"]) else np.inf
    if E_min >= E_max:
        E_min, E_max = -np.inf, np.inf

    cand = db[(db["_E"] >= E_min) & (db["_E"] < E_max)].copy()

    bt_list = parse_building_type_filter(arow.get("in.geometry_building_type_recs"))
    if bt_list:
        bt_lower = {b.lower() for b in bt_list}
        cand = cand[cand["in.geometry_building_type_recs"].astype(str).str.strip().str.lower().isin(bt_lower)]

    mask_bed = bedrooms_mask(cand["_bed"], arow.get("in.bedrooms"))
    cand = cand[mask_bed]

    hf = arow.get("in.heating_fuel")
    if isinstance(hf, str) and hf.strip().lower() != "any" and hf.strip() != "":
        cand = cand[cand["in.heating_fuel"].astype(str).str.strip().str.lower() == hf.strip().lower()]

    N = len(cand)
    if N == 0:
        # weights_out[sid] already zeros
        summary_rows.append({
            "sub_class_id": sid,
            "status": "no_candidates_after_filter",
            "solve_sec": 0.0,
            "candidates_N": 0,
            "H_target": H,
            "H_lo": np.nan, "H_hi": np.nan,
            "sum_w": 0.0,
            "housingstock_error_pct": np.nan,
            "E_min": E_min, "E_max": E_max,
            "TotalEnergy_target_kwh": float(TotE) if np.isfinite(TotE) else np.nan,
            "TotalEnergy_actual_kwh": 0.0,
            "energy_rel_error_pct": np.nan,
            "rho_used": np.nan,
            "L_min": np.nan, "L_max": np.nan,
            "U_min": np.nan, "U_max": np.nan,
            "objective_value": np.nan,
            "obj_energy": np.nan, "obj_reg": np.nan,
            "active_archetypes_w>0": 0,
            "top1_share": np.nan, "top5_share": np.nan,
            "N_eff": 0.0, "gini": 0.0,
            "bt_filter": "; ".join(bt_list) if bt_list else "any",
            "bedrooms_filter": str(arow.get("in.bedrooms")),
            "heating_fuel_filter": str(arow.get("in.heating_fuel")),
            "appliance_constraints_applied": False,
        })
        continue

    # Data vectors
    E = cand["_E"].to_numpy(dtype=float)

    # Bounds
    rho = auto_shrink_rho_for_feasibility(float(rho_min_weight), H, N)
    L   = rho * H / max(N, 1) * np.ones(N, dtype=float)
    U   = np.full(N, wmax_fraction_of_H * H, dtype=float) if (wmax_fraction_of_H is not None) else np.full(N, float('inf'))

    # Sum band
    H_lo = (1.0 - float(sum_band_eps)) * float(H)
    H_hi = (1.0 + float(sum_band_eps)) * float(H)

    # Energy per HH target
    E_star = (TotE / H) if (np.isfinite(TotE) and H > 0) else np.nan

    # Appliance constraints toggle for this subclass
    apply_appliance = is_yes(arow.get(APPLIANCE_FLAG_COL))

    # Build & solve model
    model = ConcreteModel()
    model.K = PyomoSet(initialize=list(range(N)))

    def w_bounds(m, k):
        return (float(L[k]), float(U[k]))
    model.w = Var(model.K, domain=NonNegativeReals, bounds=w_bounds)

    # Sum band constraints
    if sum_band_eps <= 0.0:
        def sum_w_eq(m):
            return sum(m.w[k] for k in m.K) == float(H)
        model.sum_w_eq = Constraint(rule=sum_w_eq)
        sum_w_expr = sum(model.w[k] for k in model.K)
    else:
        model.sum_w_lo = Constraint(expr=sum(model.w[k] for k in model.K) >= H_lo)
        model.sum_w_hi = Constraint(expr=sum(model.w[k] for k in model.K) <= H_hi)
        sum_w_expr = sum(model.w[k] for k in model.K)

    # Energy objective
    def objective_rule(m):
        energy_target = float(H) * (0.0 if not np.isfinite(E_star) else float(E_star))
        energy_dev = (sum(m.w[k] * E[k] for k in m.K) - energy_target)**2
        reg = sum(m.w[k]**2 for k in m.K)
        return float(alpha) * energy_dev + float(gamma) * reg
    model.obj = Objective(rule=objective_rule)

    # Appliance penetration constraints (linear, only if requested and active per-field)
    applied_any = False
    if apply_appliance:
        for field, cfg in APPLIANCE_CONSTRAINTS.items():
            if not cfg.get("active", False):
                continue
            t = float(cfg.get("target_pct", 0.0))
            d = float(cfg.get("tol_abs", 0.0))
            lo = max(0.0, t - d)
            hi = min(1.0, t + d)

            Ak = cand[field].to_numpy(dtype=float) if field in cand.columns else np.zeros(N, dtype=float)

            lhs_num = sum(model.w[k] * Ak[k] for k in model.K)
            model.add_component(f"appl_{re.sub(r'[^A-Za-z0-9_]+','_',field)}_lo",
                                Constraint(expr=lhs_num - lo * sum_w_expr >= 0))
            model.add_component(f"appl_{re.sub(r'[^A-Za-z0-9_]+','_',field)}_hi",
                                Constraint(expr=lhs_num - hi * sum_w_expr <= 0))
            applied_any = True

    # Solve
    try:
        res = solver.solve(model, tee=False)
        solver_status = str(res.solver.status)
        solver_term   = str(res.solver.termination_condition)
        term_ok = str(res.solver.termination_condition).lower() in {"optimal", "locallyoptimal"}
    except Exception as e:
        solver_status = "exception"
        solver_term   = f"{e}"
        term_ok = False

    if not term_ok:
        w_opt = np.zeros(N, dtype=float)  # (a) infeasible -> zeros
        status_text = f"{solver_status}/{solver_term} [applied_appliance={applied_any}] (infeasible->zeros)"
    else:
        w_opt = np.array([value(model.w[k]) if model.w[k].value is not None else 0.0 for k in model.K], dtype=float)
        status_text = f"{solver_status}/{solver_term} [applied_appliance={applied_any}]"

    # Map to full DB rows (weights_out already has zeros; overwrite column with solution mapping)
    col = np.zeros(len(db), dtype=float)
    if N > 0:
        col[cand.index.to_numpy()] = w_opt
    weights_out[sid] = col

    # Diagnostics
    sum_w = float(np.nansum(w_opt))
    TotalEnergy_actual = float(np.dot(w_opt, E)) if N > 0 else 0.0
    energy_rel_error = ((TotalEnergy_actual - TotE) / TotE * 100.0) if (np.isfinite(TotE) and TotE != 0) else np.nan
    _E_star_val = 0.0 if not np.isfinite(E_star) else float(E_star)
    obj_energy = float((np.dot(w_opt, E) - float(H) * _E_star_val)**2)
    obj_reg    = float(np.sum(w_opt ** 2))
    obj_total  = float(alpha * obj_energy + gamma * obj_reg)

    active = int(np.sum(w_opt > 0)) if N > 0 else 0
    if N > 0 and sum_w > 0:
        ws = np.sort(w_opt)[::-1]
        top1_share = float(ws[0] / sum_w)
        top5_share = float(ws[:min(5, N)].sum() / sum_w)
    else:
        top1_share = top5_share = np.nan

    housingstock_error_pct = ((sum_w - H) / H * 100.0) if (H != 0) else np.nan

    t1 = time.perf_counter()
    summary_rows.append({
        "sub_class_id": sid,
        "status": status_text,
        "solve_sec": t1 - t0,
        "candidates_N": int(N),
        "H_target": int(H),
        "H_lo": H_lo, "H_hi": H_hi,
        "sum_w": sum_w,
        "housingstock_error_pct": float(housingstock_error_pct) if np.isfinite(housingstock_error_pct) else np.nan,
        "E_min": E_min, "E_max": E_max,
        "TotalEnergy_target_kwh": float(TotE) if np.isfinite(TotE) else np.nan,
        "TotalEnergy_actual_kwh": float(TotalEnergy_actual),
        "energy_rel_error_pct": float(energy_rel_error) if np.isfinite(energy_rel_error) else np.nan,
        "rho_used": float(rho),
        "L_min": float(np.min(L)), "L_max": float(np.max(L)),
        "U_min": float(np.min(U)), "U_max": float(np.max(U)),
        "objective_value": obj_total,
        "obj_energy": obj_energy, "obj_reg": obj_reg,
        "active_archetypes_w>0": active,
        "top1_share": top1_share, "top5_share": top5_share,
        "N_eff": effective_sample_size(w_opt), "gini": gini_coef(w_opt),
        "bt_filter": "; ".join(bt_list) if bt_list else "any",
        "bedrooms_filter": str(arow.get("in.bedrooms")),
        "heating_fuel_filter": str(arow.get("in.heating_fuel")),
        "appliance_constraints_applied": bool(applied_any),
    })

# -----------------------
# SAVE TO NEW FILE
# -----------------------
stem = Path(excel_path).with_suffix("").name
default_name = f"{stem}__weights_{FROM_SUBCLASS}_to_{TO_SUBCLASS}__energy_appliance.xlsx"
out_path = Path(excel_path).with_name(save_as or default_name)

# Build normalized weights sheet (per subclass). Keep all subclasses; zeros remain zeros.
weights_norm = weights_out.copy()
for c in list(weights_norm.columns):
    if c == "bldg_id":
        continue
    s = weights_norm[c].sum()
    if s > 0:
        weights_norm[c] = weights_norm[c] / s
    # else keep zeros if no mass

with pd.ExcelWriter(out_path, engine="openpyxl", mode="w") as writer:
    weights_out.to_excel(writer, sheet_name="weights", index=False)
    weights_norm.to_excel(writer, sheet_name="weights_normalized", index=False)
    pd.DataFrame(summary_rows).to_excel(writer, sheet_name="weights_summary", index=False)

print(f"Done. Wrote results (selection-aware + optional appliance bands) to: {out_path}")


Running subclasses (full list for output): ['Subclass_01', 'Subclass_02', 'Subclass_03', 'Subclass_04', 'Subclass_05', 'Subclass_06', 'Subclass_07', 'Subclass_08', 'Subclass_09', 'Subclass_10', 'Subclass_11', 'Subclass_12', 'Subclass_13', 'Subclass_14', 'Subclass_15', 'Subclass_16', 'Subclass_17', 'Subclass_18', 'Subclass_19']
Done. Wrote results (selection-aware + optional appliance bands) to: C:\Users\dmitr\Article_02\nrel_weights\weight_solver_15i__weights_Subclass_01_to_Subclass_19__energy_appliance.xlsx


In [2]:
# =========================
# Cell — Appliance penetrations + Segment totals checks
# =========================
# Adds a new tab 'segment_totals_check' comparing:
#   - sum of non-normalized weights per subclass vs HousingStock (from 'archetypes' in excel_path)
#   - weighted energy per subclass (Σ w * E) vs annual_consumption_kwh (from 'archetypes' in excel_path)
#
# Also (as before) writes:
#   - 'appliance_penetrations_by_subclass'
#   - 'appliance_penetrations_total'

import numpy as np
import pandas as pd
from pathlib import Path

# -------- CONFIG --------
excel_path   = r"C:\Users\dmitr\Article_02\nrel_weights/weight_solver_13.xlsx"          # source workbook (has 'archetypes', 'nrel_db')
weights_path = r"C:\Users\dmitr\Article_02\nrel_weights\final_weights_04.xlsx"          # workbook with 'weights' from Cell 2

# Same 4 appliances as before
APPLIANCE_CONSTRAINTS = {
    "in.hvac_cooling_type":       {"active": True, "target_pct": 0.20, "tol_abs": 0.005},
    "in.clothes_washer_presence": {"active": True, "target_pct": 0.92, "tol_abs": 0.005},
    "in.misc_freezer":            {"active": True, "target_pct": 0.12, "tol_abs": 0.005},
    "in.dishwasher":              {"active": True, "target_pct": 0.10, "tol_abs": 0.005},
}

# -------- LOAD DATA --------
# 1) Source: nrel_db (appliance flags + energy) and archetypes (targets)
xls = pd.ExcelFile(excel_path, engine="openpyxl")
db  = pd.read_excel(xls, sheet_name="nrel_db")
arch = pd.read_excel(xls, sheet_name="archetypes")

if "bldg_id" not in db.columns:
    raise KeyError("Column 'bldg_id' is missing in nrel_db.")
if "sub_class_id" not in arch.columns:
    raise KeyError("Column 'sub_class_id' is missing in archetypes.")
if "HousingStock" not in arch.columns or "annual_consumption_kwh" not in arch.columns:
    raise KeyError("Columns 'HousingStock' and 'annual_consumption_kwh' must exist in archetypes.")

# Energy per building (coerce to float)
E_col = "out.electricity.total.energy_consumption"
if E_col not in db.columns:
    raise KeyError(f"Column '{E_col}' is missing in nrel_db.")
db["_E"] = pd.to_numeric(db[E_col], errors="coerce").fillna(0.0)

# 2) Weights
wxls = pd.ExcelFile(weights_path, engine="openpyxl")
weights_df = pd.read_excel(wxls, sheet_name="weights")
if "bldg_id" not in weights_df.columns:
    raise KeyError("Column 'bldg_id' is missing in weights sheet.")
subclass_cols = [c for c in weights_df.columns if c != "bldg_id"]
if not subclass_cols:
    raise RuntimeError("No subclass weight columns found in 'weights' sheet.")

# -------- PREPARE APPLIANCE FLAGS --------
appliance_fields = list(APPLIANCE_CONSTRAINTS.keys())
for f in appliance_fields:
    if f in db.columns:
        col = pd.to_numeric(db[f], errors="coerce").fillna(0.0)
        db[f] = (col != 0).astype(int)  # force to {0,1}
    else:
        db[f] = 0

# Align weights to db by bldg_id for all downstream calcs
merged = weights_df.merge(db[["bldg_id", "_E"] + appliance_fields], on="bldg_id", how="left")
merged["_E"] = merged["_E"].fillna(0.0)
for f in appliance_fields:
    merged[f] = merged[f].fillna(0).astype(int)

# -------- (A) PER-SUBCLASS APPLIANCE PENETRATIONS --------
rows = []
for f in appliance_fields:
    row = {}
    A = merged[f].to_numpy(dtype=float)
    for sc in subclass_cols:
        w = pd.to_numeric(merged[sc], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        denom = float(w.sum())
        row[sc] = float((w * A).sum() / denom) if denom > 0 else np.nan
    rows.append((f, row))
by_subclass = pd.DataFrame({f: r for f, r in rows}).T
by_subclass.index.name = "field"

# -------- (B) TOTAL-SAMPLE APPLIANCE PENETRATIONS --------
total_rows = []
for f in appliance_fields:
    num = 0.0
    den = 0.0
    A = merged[f].to_numpy(dtype=float)
    for sc in subclass_cols:
        w = pd.to_numeric(merged[sc], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        num += float((w * A).sum())
        den += float(w.sum())
    pen = (num / den) if den > 0 else np.nan
    target = float(APPLIANCE_CONSTRAINTS[f].get("target_pct", np.nan))
    tol = float(APPLIANCE_CONSTRAINTS[f].get("tol_abs", np.nan))
    total_rows.append({
        "field": f,
        "penetration": pen,
        "target_pct": target,
        "tol_abs": tol,
        "within_band": (abs(pen - target) <= tol) if (np.isfinite(pen) and np.isfinite(target) and np.isfinite(tol)) else np.nan,
        "penetration_pct": (pen * 100.0) if np.isfinite(pen) else np.nan,
        "target_pct_pct": (target * 100.0) if np.isfinite(target) else np.nan,
        "tol_abs_pct_points": (tol * 100.0) if np.isfinite(tol) else np.nan,
    })
total_df = pd.DataFrame(
    total_rows,
    columns=["field","penetration","target_pct","tol_abs","within_band","penetration_pct","target_pct_pct","tol_abs_pct_points"]
)

# -------- (C) NEW: SEGMENT TOTALS CHECK (Σw vs HousingStock; Σ w·E vs annual_consumption_kwh) --------
# Prepare targets by subclass
arch_tgt = (arch[["sub_class_id", "HousingStock", "annual_consumption_kwh"]]
            .drop_duplicates(subset=["sub_class_id"])
            .set_index("sub_class_id"))

# Compute sums from weights and weighted energy using _E
seg_rows = []
for sc in subclass_cols:
    w = pd.to_numeric(merged[sc], errors="coerce").fillna(0.0).to_numpy(dtype=float)
    sum_w = float(w.sum())
    sum_E = float((w * merged["_E"].to_numpy(dtype=float)).sum())

    H_target = float(arch_tgt.loc[sc, "HousingStock"]) if sc in arch_tgt.index and pd.notna(arch_tgt.loc[sc, "HousingStock"]) else np.nan
    TotE_target = float(arch_tgt.loc[sc, "annual_consumption_kwh"]) if sc in arch_tgt.index and pd.notna(arch_tgt.loc[sc, "annual_consumption_kwh"]) else np.nan

    # diffs
    H_abs_diff = (sum_w - H_target) if np.isfinite(H_target) else np.nan
    H_rel_diff_pct = ((sum_w - H_target) / H_target * 100.0) if (np.isfinite(H_target) and H_target != 0) else np.nan

    E_abs_diff = (sum_E - TotE_target) if np.isfinite(TotE_target) else np.nan
    E_rel_diff_pct = ((sum_E - TotE_target) / TotE_target * 100.0) if (np.isfinite(TotE_target) and TotE_target != 0) else np.nan

    seg_rows.append({
        "sub_class_id": sc,
        "H_target": H_target,
        "sum_w": sum_w,
        "H_abs_diff": H_abs_diff,
        "H_rel_diff_pct": H_rel_diff_pct,
        "TotalEnergy_target_kwh": TotE_target,
        "TotalEnergy_actual_kwh": sum_E,
        "energy_abs_diff": E_abs_diff,
        "energy_rel_diff_pct": E_rel_diff_pct,
    })

segment_totals_check = pd.DataFrame(seg_rows, columns=[
    "sub_class_id",
    "H_target","sum_w","H_abs_diff","H_rel_diff_pct",
    "TotalEnergy_target_kwh","TotalEnergy_actual_kwh","energy_abs_diff","energy_rel_diff_pct"
])

# Optional totals row
tot_H_target = arch_tgt.reindex(subclass_cols)["HousingStock"].sum(min_count=1)
tot_H_sumw   = segment_totals_check["sum_w"].sum(min_count=1)
tot_TotE_tgt = arch_tgt.reindex(subclass_cols)["annual_consumption_kwh"].sum(min_count=1)
tot_TotE_act = segment_totals_check["TotalEnergy_actual_kwh"].sum(min_count=1)
segment_totals_check.loc[len(segment_totals_check)] = {
    "sub_class_id": "__TOTAL__",
    "H_target": tot_H_target,
    "sum_w": tot_H_sumw,
    "H_abs_diff": (tot_H_sumw - tot_H_target) if np.isfinite(tot_H_target) else np.nan,
    "H_rel_diff_pct": ((tot_H_sumw - tot_H_target)/tot_H_target*100.0) if (np.isfinite(tot_H_target) and tot_H_target!=0) else np.nan,
    "TotalEnergy_target_kwh": tot_TotE_tgt,
    "TotalEnergy_actual_kwh": tot_TotE_act,
    "energy_abs_diff": (tot_TotE_act - tot_TotE_tgt) if np.isfinite(tot_TotE_tgt) else np.nan,
    "energy_rel_diff_pct": ((tot_TotE_act - tot_TotE_tgt)/tot_TotE_tgt*100.0) if (np.isfinite(tot_TotE_tgt) and tot_TotE_tgt!=0) else np.nan,
}

# -------- WRITE OUTPUT TABS INTO THE SAME WEIGHTS FILE --------
with pd.ExcelWriter(weights_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    by_subclass.to_excel(writer, sheet_name="appliance_penetrations_by_subclass")
    total_df.to_excel(writer, sheet_name="appliance_penetrations_total", index=False)
    segment_totals_check.to_excel(writer, sheet_name="segment_totals_check", index=False)

print(f"Done. Wrote 'appliance_penetrations_by_subclass', 'appliance_penetrations_total', and 'segment_totals_check' to: {weights_path}")


C:\Users\dmitr\AppData\Local\Programs\Python\Python310\lib\site-packages\openpyxl\workbook\child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")


Done. Wrote 'appliance_penetrations_by_subclass', 'appliance_penetrations_total', and 'segment_totals_check' to: C:\Users\dmitr\Article_02\nrel_weights\final_weights_04.xlsx


In [7]:
# =========================
# Cell — Appliance penetrations + Segment totals + Weight stats
# =========================
# Inputs:
#   - excel_path:  source workbook with 'archetypes' and 'nrel_db'
#   - weights_path: workbook with sheets:
#       * 'weights'            (non-normalized weights per subclass + Overall)
#       * 'weights_normalized' (column-normalized 0..1 per subclass + Overall)
#
# Outputs (written into the SAME weights file):
#   - 'appliance_penetrations_by_subclass'
#   - 'appliance_penetrations_total'
#   - 'segment_totals_check'
#   - 'weight_stats_by_segment'

import numpy as np
import pandas as pd
from pathlib import Path

# -------- CONFIG --------
excel_path   = r"C:\Users\dmitr\Article_02\nrel_weights/weight_solver_13.xlsx"
weights_path = r"C:\Users\dmitr\Article_02\nrel_weights\final_weights_05c.xlsx"

APPLIANCE_CONSTRAINTS = {
    "in.hvac_cooling_type":       {"active": True, "target_pct": 0.20, "tol_abs": 0.005},
    "in.clothes_washer_presence": {"active": True, "target_pct": 0.92, "tol_abs": 0.005},
    "in.misc_freezer":            {"active": True, "target_pct": 0.12, "tol_abs": 0.005},
    "in.dishwasher":              {"active": True, "target_pct": 0.10, "tol_abs": 0.005},
}

# -------- HELPERS --------
def gini_coef(arr):
    """Gini coefficient for a 1D array of weights (zeros allowed)."""
    x = np.asarray(arr, dtype=float)
    if x.size == 0:
        return np.nan
    if np.allclose(x, 0):
        return 0.0
    x_sorted = np.sort(x)
    n = x_sorted.size
    cum = np.cumsum(x_sorted)
    return float((n + 1 - 2.0 * np.sum(cum) / cum[-1]) / n)

# -------- LOAD SOURCE DATA (nrel_db + archetypes) --------
xls = pd.ExcelFile(excel_path, engine="openpyxl")
db  = pd.read_excel(xls, sheet_name="nrel_db")
arch = pd.read_excel(xls, sheet_name="archetypes")

if "bldg_id" not in db.columns:
    raise KeyError("Column 'bldg_id' is missing in nrel_db.")
if "sub_class_id" not in arch.columns:
    raise KeyError("Column 'sub_class_id' is missing in archetypes.")
if "HousingStock" not in arch.columns or "annual_consumption_kwh" not in arch.columns:
    raise KeyError("Columns 'HousingStock' and 'annual_consumption_kwh' must exist in archetypes.")

E_col = "out.electricity.total.energy_consumption"
if E_col not in db.columns:
    raise KeyError(f"Column '{E_col}' is missing in nrel_db.")
db["_E"] = pd.to_numeric(db[E_col], errors="coerce").fillna(0.0)

# -------- LOAD WEIGHTS & NORMALIZED WEIGHTS --------
wxls = pd.ExcelFile(weights_path, engine="openpyxl")
weights_df = pd.read_excel(wxls, sheet_name="weights")
if "bldg_id" not in weights_df.columns:
    raise KeyError("Column 'bldg_id' is missing in 'weights' sheet.")
subclass_cols = [c for c in weights_df.columns if c != "bldg_id"]
if not subclass_cols:
    raise RuntimeError("No subclass weight columns found in 'weights' sheet.")

# normalized weights
weights_norm_df = pd.read_excel(wxls, sheet_name="weights_normalized")
if "bldg_id" not in weights_norm_df.columns:
    raise KeyError("Column 'bldg_id' is missing in 'weights_normalized' sheet.")

# ensure same bldg_id order for both weight tables
if not weights_norm_df["bldg_id"].equals(weights_df["bldg_id"]):
    # align normalized weights to weights_df order
    weights_norm_df = weights_norm_df.set_index("bldg_id").reindex(weights_df["bldg_id"]).reset_index()

# detect Overall column if present
has_overall = "Overall" in weights_df.columns and "Overall" in weights_norm_df.columns
segment_cols_for_stats = subclass_cols.copy()
if has_overall:
    segment_cols_for_stats.append("Overall")

# -------- PREPARE APPLIANCE FLAGS & MERGE WITH WEIGHTS --------
appliance_fields = list(APPLIANCE_CONSTRAINTS.keys())
for f in appliance_fields:
    if f in db.columns:
        col = pd.to_numeric(db[f], errors="coerce").fillna(0.0)
        db[f] = (col != 0).astype(int)
    else:
        db[f] = 0

merged = weights_df.merge(db[["bldg_id", "_E"] + appliance_fields],
                          on="bldg_id", how="left")
merged["_E"] = merged["_E"].fillna(0.0)
for f in appliance_fields:
    merged[f] = merged[f].fillna(0).astype(int)

# -------- (A) PER-SUBCLASS APPLIANCE PENETRATIONS --------
rows = []
for f in appliance_fields:
    row = {}
    A = merged[f].to_numpy(dtype=float)
    for sc in subclass_cols:
        w = pd.to_numeric(merged[sc], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        denom = float(w.sum())
        row[sc] = float((w * A).sum() / denom) if denom > 0 else np.nan
    rows.append((f, row))

by_subclass = pd.DataFrame({f: r for f, r in rows}).T
by_subclass.index.name = "field"

# -------- (B) TOTAL-SAMPLE APPLIANCE PENETRATIONS --------
total_rows = []
for f in appliance_fields:
    num = 0.0
    den = 0.0
    A = merged[f].to_numpy(dtype=float)
    for sc in subclass_cols:
        w = pd.to_numeric(merged[sc], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        num += float((w * A).sum())
        den += float(w.sum())
    pen = (num / den) if den > 0 else np.nan
    target = float(APPLIANCE_CONSTRAINTS[f].get("target_pct", np.nan))
    tol = float(APPLIANCE_CONSTRAINTS[f].get("tol_abs", np.nan))
    total_rows.append({
        "field": f,
        "penetration": pen,
        "target_pct": target,
        "tol_abs": tol,
        "within_band": (abs(pen - target) <= tol) if (np.isfinite(pen) and np.isfinite(target) and np.isfinite(tol)) else np.nan,
        "penetration_pct": (pen * 100.0) if np.isfinite(pen) else np.nan,
        "target_pct_pct": (target * 100.0) if np.isfinite(target) else np.nan,
        "tol_abs_pct_points": (tol * 100.0) if np.isfinite(tol) else np.nan,
    })

total_df = pd.DataFrame(
    total_rows,
    columns=["field","penetration","target_pct","tol_abs","within_band",
             "penetration_pct","target_pct_pct","tol_abs_pct_points"]
)

# -------- (C) SEGMENT TOTALS CHECK (Σw vs H; Σw·E vs TotE) --------
arch_tgt = (arch[["sub_class_id", "HousingStock", "annual_consumption_kwh"]]
            .drop_duplicates(subset=["sub_class_id"])
            .set_index("sub_class_id"))

seg_rows = []
for sc in subclass_cols:
    w = pd.to_numeric(merged[sc], errors="coerce").fillna(0.0).to_numpy(dtype=float)
    sum_w = float(w.sum())
    sum_E = float((w * merged["_E"].to_numpy(dtype=float)).sum())

    H_target = float(arch_tgt.loc[sc, "HousingStock"]) if sc in arch_tgt.index and pd.notna(arch_tgt.loc[sc, "HousingStock"]) else np.nan
    TotE_target = float(arch_tgt.loc[sc, "annual_consumption_kwh"]) if sc in arch_tgt.index and pd.notna(arch_tgt.loc[sc, "annual_consumption_kwh"]) else np.nan

    H_abs_diff = (sum_w - H_target) if np.isfinite(H_target) else np.nan
    H_rel_diff_pct = ((sum_w - H_target) / H_target * 100.0) if (np.isfinite(H_target) and H_target != 0) else np.nan

    E_abs_diff = (sum_E - TotE_target) if np.isfinite(TotE_target) else np.nan
    E_rel_diff_pct = ((sum_E - TotE_target) / TotE_target * 100.0) if (np.isfinite(TotE_target) and TotE_target != 0) else np.nan

    seg_rows.append({
        "sub_class_id": sc,
        "H_target": H_target,
        "sum_w": sum_w,
        "H_abs_diff": H_abs_diff,
        "H_rel_diff_pct": H_rel_diff_pct,
        "TotalEnergy_target_kwh": TotE_target,
        "TotalEnergy_actual_kwh": sum_E,
        "energy_abs_diff": E_abs_diff,
        "energy_rel_diff_pct": E_rel_diff_pct,
    })

segment_totals_check = pd.DataFrame(seg_rows, columns=[
    "sub_class_id",
    "H_target","sum_w","H_abs_diff","H_rel_diff_pct",
    "TotalEnergy_target_kwh","TotalEnergy_actual_kwh",
    "energy_abs_diff","energy_rel_diff_pct"
])

# Optional overall row
tot_H_target = arch_tgt.reindex(subclass_cols)["HousingStock"].sum(min_count=1)
tot_H_sumw   = segment_totals_check["sum_w"].sum(min_count=1)
tot_TotE_tgt = arch_tgt.reindex(subclass_cols)["annual_consumption_kwh"].sum(min_count=1)
tot_TotE_act = segment_totals_check["TotalEnergy_actual_kwh"].sum(min_count=1)

segment_totals_check.loc[len(segment_totals_check)] = {
    "sub_class_id": "__TOTAL__",
    "H_target": tot_H_target,
    "sum_w": tot_H_sumw,
    "H_abs_diff": (tot_H_sumw - tot_H_target) if np.isfinite(tot_H_target) else np.nan,
    "H_rel_diff_pct": ((tot_H_sumw - tot_H_target)/tot_H_target*100.0)
                       if (np.isfinite(tot_H_target) and tot_H_target!=0) else np.nan,
    "TotalEnergy_target_kwh": tot_TotE_tgt,
    "TotalEnergy_actual_kwh": tot_TotE_act,
    "energy_abs_diff": (tot_TotE_act - tot_TotE_tgt) if np.isfinite(tot_TotE_tgt) else np.nan,
    "energy_rel_diff_pct": ((tot_TotE_act - tot_TotE_tgt)/tot_TotE_tgt*100.0)
                           if (np.isfinite(tot_TotE_tgt) and tot_TotE_tgt!=0) else np.nan,
}

# -------- (D) NEW: WEIGHT STATS BY SEGMENT (normalized weights) --------
# For each segment (subclass) and Overall: count of non-zero normalized weights,
# max / min normalized weight, and Gini coefficient of normalized weights.

stats_rows = []
for seg in segment_cols_for_stats:
    if seg not in weights_norm_df.columns:
        continue
    w_norm = pd.to_numeric(weights_norm_df[seg], errors="coerce").fillna(0.0).to_numpy(dtype=float)

    nnz_mask = w_norm > 0
    n_nonzero = int(nnz_mask.sum())

    if n_nonzero > 0:
        max_norm = float(w_norm[nnz_mask].max())
        min_norm = float(w_norm[nnz_mask].min())
        gini = gini_coef(w_norm)  # gini over full distribution incl. zeros
    else:
        max_norm = np.nan
        min_norm = np.nan
        gini = 0.0  # all zero → no dispersion

    stats_rows.append({
        "segment_id": seg,
        "n_profiles_nonzero": n_nonzero,
        "max_normalized_weight": max_norm,
        "min_normalized_weight": min_norm,
        "gini_normalized_weights": gini,
    })

weight_stats_by_segment = pd.DataFrame(
    stats_rows,
    columns=["segment_id","n_profiles_nonzero",
             "max_normalized_weight","min_normalized_weight",
             "gini_normalized_weights"]
)

# -------- WRITE OUTPUT TABS INTO THE SAME WEIGHTS FILE --------
with pd.ExcelWriter(weights_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    by_subclass.to_excel(writer, sheet_name="appliance_penetrations_by_subclass")
    total_df.to_excel(writer, sheet_name="appliance_penetrations_total", index=False)
    segment_totals_check.to_excel(writer, sheet_name="segment_totals_check", index=False)
    weight_stats_by_segment.to_excel(writer, sheet_name="weight_stats_by_segment", index=False)

print(
    "Done. Wrote 'appliance_penetrations_by_subclass', "
    "'appliance_penetrations_total', 'segment_totals_check', "
    "and 'weight_stats_by_segment' to:", weights_path
)


C:\Users\dmitr\AppData\Local\Programs\Python\Python310\lib\site-packages\openpyxl\workbook\child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")


Done. Wrote 'appliance_penetrations_by_subclass', 'appliance_penetrations_total', 'segment_totals_check', and 'weight_stats_by_segment' to: C:\Users\dmitr\Article_02\nrel_weights\final_weights_05c.xlsx
